# Whisper streaming policies: Indonesian vs English (FLEURS)

Benchmark with the analysis fixed before the runs (Chapter Plan + review round 1 fixes R1, R2, R3, R5).

**Kaggle settings (right-hand panel):** Accelerator **GPU T4 x2**, Internet **On**, Persistence: none needed.

**How to run**
1. (Done on 22 Oct 2026.) Check session: set `MODE = "check"` and use **Save Version → Save & Run All (Commit)**.
   It installs everything, runs the unit tests, the QA checks (harness = released tools),
   every job on 2 sentence indices, the full analysis on that mini set, and prints
   the GPU-hour estimate for the full plan. Send all the output back.
2. Main runs: `MODE = "run"` (the default) and commit. Each session runs jobs in priority order until
   `SESSION_HOURS`, saves every finished recording, and writes the report on whatever is done.
3. To continue in a new session: **Add Input → Notebook Output → this notebook's previous
   version**, then commit again. Finished recordings are restored and skipped.

**Two accounts (you and a second person, each on their own account):** set `N_SHARDS = 2` and
`SHARD_INDEX = 0` on one account, `SHARD_INDEX = 1` on the other. Each runs its own half of the jobs
(the split is computed the same way on both). When both halves are done, the second person downloads
the zip from the last cell and gives it to you; upload it as a private Kaggle **Dataset**, add it
(and your own previous output) as input, and commit once more: the report then covers all jobs.

All outputs go to `/kaggle/working` (results, logs, report, figure). Data and models live in
`/tmp` and are downloaded again each session.

In [ ]:
import os, sys, time
T0 = time.time()

MODE = "run"              # "run" (main runs; the check run was done on 22 Oct 2026) or "check"
SESSION_HOURS = 11.0      # stop starting new recordings after this (Kaggle limit is 12 h)
REPORT_RESERVE_MIN = 25   # time kept for the final report
CHECK_N_INDICES = 2       # sentence indices per job in check mode

N_SHARDS = 2              # number of accounts running the jobs (1 = this account runs everything)
SHARD_INDEX = 0           # 0 = first account, 1 = second account (must differ between accounts)

MODEL = "large-v3"
PRIMARY_AA = "aa_trim"    # always holds the last word back (--never_fire); the as-released default
                          # (aa_keep) degenerated into "." loops at 0.5/1.0 s in the check run and is
                          # reported as "SimulStreaming as released"
HOLD_N = 2
FRAME_THRESHOLD = 25
EQUIVALENCE_MARGIN = 0.42 # fixed before the runs: +/-42 points of relative advantage (review R2)
BOOTSTRAP_B = 10000
SEED = 20261022
VAD_PARAMS = dict(threshold=0.5, min_silence_duration_ms=500, speech_pad_ms=100)

OUT = "/kaggle/working"
WORK = "/tmp/wspb_work"
SS_DIR = f"{WORK}/SimulStreaming"
WS_DIR = f"{WORK}/whisper_streaming"
DATA_DIR = f"{WORK}/fleurs"
MODEL_DIR = f"{WORK}/models"
STATE = f"{OUT}/state"
RESULTS = f"{OUT}/results" if MODE == "run" else f"{OUT}/check_results"
REPORT = f"{OUT}/report" if MODE == "run" else f"{OUT}/check_report"
for d in (WORK, MODEL_DIR, STATE, RESULTS, REPORT, f"{OUT}/wspb"):
    os.makedirs(d, exist_ok=True)
print("MODE:", MODE)

## Install and pinned code

In [ ]:
import subprocess

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"failed: {cmd}")
    return r.stdout

sh("pip install -q openai-whisper==20250625 silero-vad==5.1.2 faster-whisper==1.1.1 rapidfuzz tabulate")
if not os.path.exists(SS_DIR):
    sh(f"git clone -q https://github.com/ufal/SimulStreaming {SS_DIR}")
sh(f"git -C {SS_DIR} checkout -q 077ea37d5ab4ff98bc567e4507f140dc4e5d5ad6")
if not os.path.exists(WS_DIR):
    sh(f"git clone -q https://github.com/ufal/whisper_streaming {WS_DIR}")
sh(f"git -C {WS_DIR} checkout -q 6da90b44b7e50d79695e68166d2a2c7609c75abb")
print(sh(f"git -C {SS_DIR} log -1 --format='SimulStreaming %H'"), end="")
print(sh(f"git -C {WS_DIR} log -1 --format='whisper_streaming %H'"), end="")

# cuDNN 9 from the pip wheel, for faster-whisper (anchor runs) in the worker processes
import glob, site
CUDNN = [p for sp in site.getsitepackages() for p in glob.glob(f"{sp}/nvidia/cudnn/lib")]
print("cuDNN lib:", CUDNN)

## The `wspb` package (one cell per module)

In [ ]:
os.makedirs("/kaggle/working/wspb", exist_ok=True)

In [ ]:
%%writefile /kaggle/working/wspb/__init__.py
"""Whisper streaming-policy benchmark (Indonesian vs English, FLEURS).

Modules
-------
common   paths, config and small I/O helpers
data     FLEURS download, paired manifest, audio loading, VAD statistics
text     Whisper text normalisers and number-word detection
align    word-level Levenshtein alignment
commit   commit rules of the re-decoding policies (hold-n, LocalAgreement-2)
engines  model wrappers: re-decoding Whisper, AlignAtt (SimulStreaming), offline, anchor
sim      streaming simulation (computation-unaware and computation-aware)
worker   job runner (one process per GPU, resumable)
metrics  per-recording metrics (WER, LAAL, erasure, tail confirmation, ...)
stats    planned tests (H1-H5, M1, M3, M4), matched-latency H3, and bootstrap
qa       QA before the main runs: the harness reproduces Whisper-Streaming and SimulStreaming
budget   GPU-time estimate for the full job list from a check run
report   tables, Figure 1 and a summary
"""

In [ ]:
%%writefile /kaggle/working/wspb/common.py
import json
import os
import sys
import time
from pathlib import Path

SR = 16000

# Pinned versions of the code and data used by the benchmark.
SIMULSTREAMING_COMMIT = "077ea37d5ab4ff98bc567e4507f140dc4e5d5ad6"
WHISPER_STREAMING_COMMIT = "6da90b44b7e50d79695e68166d2a2c7609c75abb"
FLEURS_REVISION = "70bb2e84b976b7e960aa89f1c648e09c59f894dd"

# Short language code -> FLEURS config and Whisper language code.
LANGS = {"id": "id_id", "en": "en_us"}


def load_config(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path):
    out = []
    if not os.path.exists(path):
        return out
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                out.append(json.loads(line))
            except json.JSONDecodeError:
                # a line cut off by a killed session; the recording is simply redone
                pass
    return out


def append_jsonl(obj, path):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())


def add_simulstreaming_to_path(cfg):
    p = str(Path(cfg["simulstreaming_dir"]).resolve())
    if p not in sys.path:
        sys.path.insert(0, p)


class Timer:
    """Wall-clock timer that waits for queued GPU work before reading the clock."""

    def __init__(self, cuda):
        self.cuda = cuda

    def _sync(self):
        if self.cuda:
            import torch
            torch.cuda.synchronize()

    def __enter__(self):
        self._sync()
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        self._sync()
        self.elapsed = time.perf_counter() - self.t0
        return False


def log(msg, file=None):
    line = time.strftime("%H:%M:%S ") + str(msg)
    print(line, flush=True)
    if file:
        with open(file, "a", encoding="utf-8") as f:
            f.write(line + "\n")


def env_info():
    """Hardware and software of the run (reported with the computation-aware latencies)."""
    import platform
    info = {"python": platform.python_version(), "platform": platform.platform()}
    try:
        import torch
        info["torch"] = torch.__version__
        info["cuda"] = torch.version.cuda
        info["gpus"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    except ImportError:
        pass
    from importlib.metadata import PackageNotFoundError, version
    for pkg in ("numpy", "pandas", "scipy", "faster-whisper", "ctranslate2", "silero-vad",
                "rapidfuzz", "librosa", "soundfile", "huggingface_hub"):
        try:
            info[pkg] = version(pkg)
        except PackageNotFoundError:
            info[pkg] = None
    info.update(simulstreaming_commit=SIMULSTREAMING_COMMIT,
                whisper_streaming_commit=WHISPER_STREAMING_COMMIT, fleurs_revision=FLEURS_REVISION)
    return info

In [ ]:
%%writefile /kaggle/working/wspb/text.py
"""Text normalisation (Whisper's normalisers), CER, split/merge errors, and number detection.

Normalisation rule (fixed before the runs, review R5): English uses EnglishTextNormalizer, Indonesian
BasicTextNormalizer (as in Radford et al., 2023). BasicTextNormalizer replaces every punctuation
mark, hyphens included, with a space, so a reduplicated word ("anak-anak") is two words in the
reference and in the hypothesis alike; the same rule applies to both sides and to both languages
for erasure, tail confirmation and M4. Word-segmentation differences ("dirumah" / "di rumah") are
not normalised away: they are counted by split_merge() and are absent from CER, which is computed
on the normalised text with spaces removed (segmentation-insensitive).
"""
import re

# OpenAI Whisper's package (SimulStreaming's bundled copy lacks english.json, the spelling table
# EnglishTextNormalizer needs)
from whisper.normalizers import BasicTextNormalizer, EnglishTextNormalizer

_EN = EnglishTextNormalizer()
_BASIC = BasicTextNormalizer()


def norm_wer(text, lang):
    """Normaliser used for WER: English -> EnglishTextNormalizer, Indonesian -> BasicTextNormalizer
    (as in Radford et al., 2023)."""
    return (_EN if lang == "en" else _BASIC)(text)


def norm_basic(text):
    """Language-neutral normaliser (lowercase, punctuation and symbols removed), used for
    erasure, tail confirmation and M4 in both languages."""
    return _BASIC(text)


def words_wer(text, lang):
    return norm_wer(text, lang).split()


def words_basic(text):
    return norm_basic(text).split()


def _levenshtein(a, b):
    try:
        from rapidfuzz.distance import Levenshtein
        return Levenshtein.distance(a, b)
    except ImportError:
        prev = list(range(len(b) + 1))
        for i, ca in enumerate(a, 1):
            cur = [i]
            for j, cb in enumerate(b, 1):
                cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
            prev = cur
        return prev[-1]


def cer_counts(ref, hyp, lang):
    """Character edits and reference characters, on the WER-normalised text without spaces."""
    r = norm_wer(ref, lang).replace(" ", "")
    h = norm_wer(hyp, lang).replace(" ", "")
    return _levenshtein(r, h), len(r)


def split_merge(ops, ref, hyp):
    """Word-segmentation errors in an alignment: a split (one reference word written as two
    hypothesis words, "dirumah" -> "di rumah") or a merge (two reference words written as one).
    Each shows up as a substitution next to an insertion or deletion."""
    splits = merges = 0
    for k, (op, i, j) in enumerate(ops):
        if op != "S":
            continue
        for nb in (ops[k - 1] if k else None, ops[k + 1] if k + 1 < len(ops) else None):
            if nb is None:
                continue
            nop, ni, nj = nb
            if nop == "I":
                pair = (hyp[j] + hyp[nj]) if nj > j else (hyp[nj] + hyp[j])
                if pair == ref[i]:
                    splits += 1
                    break
            elif nop == "D":
                pair = (ref[i] + ref[ni]) if ni > i else (ref[ni] + ref[i])
                if pair == hyp[j]:
                    merges += 1
                    break
    return splits, merges


# Cardinal number words for the number-word sensitivity check (word lists fixed before the runs).
NUMBER_WORDS = {
    "en": {
        "zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten",
        "eleven", "twelve", "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen",
        "nineteen", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety",
        "hundred", "hundreds", "thousand", "thousands", "million", "millions", "billion",
        "billions", "trillion", "dozen",
    },
    "id": {
        "nol", "satu", "dua", "tiga", "empat", "lima", "enam", "tujuh", "delapan", "sembilan",
        "sepuluh", "sebelas", "belas", "puluh", "puluhan", "seratus", "ratus", "ratusan",
        "seribu", "ribu", "ribuan", "sejuta", "juta", "jutaan", "miliar", "milyar", "triliun",
        "lusin",
    },
}

_DIGIT = re.compile(r"\d")


def has_number(text, lang):
    if _DIGIT.search(text):
        return True
    return any(w in NUMBER_WORDS[lang] for w in norm_basic(text).split())


def self_test():
    from .align import align
    ref, hyp = "saya di rumah anak anak".split(), "saya dirumah anak anak".split()
    assert split_merge(align(ref, hyp), ref, hyp) == (0, 1)
    ref, hyp = "pergi kerumah".split(), "pergi ke rumah".split()
    assert split_merge(align(ref, hyp), ref, hyp) == (1, 0)
    assert norm_basic("Anak-anak") == "anak anak"
    assert cer_counts("di rumah", "dirumah", "id") == (0, 7)
    return True

In [ ]:
%%writefile /kaggle/working/wspb/align.py
"""Word-level Levenshtein alignment (unit costs) with the edit operations."""


def align(ref, hyp):
    """Return a list of (op, ref_index, hyp_index); op in C (correct), S, D, I.
    Ties are broken towards C/S, then D, then I, so the error counts equal the edit distance."""
    n, m = len(ref), len(hyp)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        d[i][0] = i
    for j in range(1, m + 1):
        d[0][j] = j
    for i in range(1, n + 1):
        ri = ref[i - 1]
        row, prev = d[i], d[i - 1]
        for j in range(1, m + 1):
            sub = prev[j - 1] + (ri != hyp[j - 1])
            row[j] = min(sub, prev[j] + 1, row[j - 1] + 1)
    ops = []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i][j] == d[i - 1][j - 1] + (ref[i - 1] != hyp[j - 1]):
            ops.append(("C" if ref[i - 1] == hyp[j - 1] else "S", i - 1, j - 1))
            i, j = i - 1, j - 1
        elif i > 0 and d[i][j] == d[i - 1][j] + 1:
            ops.append(("D", i - 1, None))
            i -= 1
        else:
            ops.append(("I", None, j - 1))
            j -= 1
    ops.reverse()
    return ops


def counts(ops):
    c = {"C": 0, "S": 0, "D": 0, "I": 0}
    for op, _, _ in ops:
        c[op] += 1
    return c

In [ ]:
%%writefile /kaggle/working/wspb/data.py
"""FLEURS test sets: download, paired manifest, audio, and data statistics."""
import csv
import tarfile
from collections import defaultdict
from pathlib import Path

import numpy as np

from .common import FLEURS_REVISION, LANGS, SR, load_json, save_json
from .text import has_number, norm_basic


def fetch_fleurs(data_dir):
    """Download test.tsv and test audio for both languages and extract the audio."""
    from huggingface_hub import hf_hub_download

    data_dir = Path(data_dir)
    for code in LANGS.values():
        for f in (f"data/{code}/test.tsv", f"data/{code}/audio/test.tar.gz"):
            hf_hub_download("google/fleurs", f, repo_type="dataset",
                            revision=FLEURS_REVISION, local_dir=str(data_dir))
        out = data_dir / "data" / code / "audio" / "test"
        marker = out / ".extracted"
        if not marker.exists():
            out.mkdir(parents=True, exist_ok=True)
            with tarfile.open(data_dir / "data" / code / "audio" / "test.tar.gz") as tar:
                tar.extractall(out)
            marker.touch()


def read_tsv(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for r in csv.reader(f, delimiter="\t", quoting=csv.QUOTE_NONE):
            rows.append({"idx": int(r[0]), "file": r[1], "ref": r[2],
                         "samples": int(r[5]), "gender": r[6]})
    return rows


def build_manifest(data_dir, max_dur=30.0):
    """Paired set: sentence indices present in both languages, minus any index with a
    recording longer than max_dur in either language (exclusion applies to both)."""
    data_dir = Path(data_dir)
    rows, wavs = {}, {}
    for lang, code in LANGS.items():
        rows[lang] = read_tsv(data_dir / "data" / code / "test.tsv")
        wavs[lang] = {p.name: str(p) for p in (data_dir / "data" / code / "audio" / "test").rglob("*.wav")}
        missing = [r["file"] for r in rows[lang] if r["file"] not in wavs[lang]]
        if missing:
            raise FileNotFoundError(f"{lang}: {len(missing)} wav files missing, e.g. {missing[:3]}")

    ids = {lang: {r["idx"] for r in rows[lang]} for lang in LANGS}
    common = ids["id"] & ids["en"]
    too_long = {r["idx"] for lang in LANGS for r in rows[lang]
                if r["idx"] in common and r["samples"] / SR > max_dur}
    numbers = {r["idx"] for lang in LANGS for r in rows[lang]
               if r["idx"] in common and has_number(r["ref"], lang)}
    paired_ids = sorted(common - too_long)

    def rec(lang, r):
        return {"lang": lang, "idx": r["idx"], "file": r["file"], "path": wavs[lang][r["file"]],
                "ref": r["ref"], "dur": r["samples"] / SR, "gender": r["gender"],
                "key": f"{lang}:{r['file']}"}

    paired = [rec(lang, r) for lang in LANGS for r in rows[lang] if r["idx"] in set(paired_ids)]
    full = [rec(lang, r) for lang in LANGS for r in rows[lang]]
    manifest = {
        "fleurs_revision": FLEURS_REVISION,
        "paired_ids": paired_ids,
        "excluded_too_long": sorted(too_long),
        "number_ids": sorted(numbers & set(paired_ids)),
        "paired": paired,
        "full": full,
    }
    save_json(manifest, data_dir / "manifest.json")
    return manifest


def load_manifest(data_dir):
    return load_json(Path(data_dir) / "manifest.json")


def load_audio(path):
    import soundfile as sf
    a, sr = sf.read(path, dtype="float32", always_2d=False)
    if a.ndim > 1:
        a = a.mean(axis=1)
    if sr != SR:
        raise ValueError(f"{path}: sample rate {sr}, expected {SR}")
    return np.ascontiguousarray(a, dtype=np.float32)


def speech_durations(manifest, out_path, vad_params):
    """Speech-only duration of every paired recording with Silero VAD (fixed parameters)."""
    import torch
    from silero_vad import get_speech_timestamps, load_silero_vad

    out_path = Path(out_path)
    done = load_json(out_path) if out_path.exists() else {}
    model = load_silero_vad()
    for r in manifest["paired"]:
        if r["key"] in done:
            continue
        wav = torch.from_numpy(load_audio(r["path"]))
        ts = get_speech_timestamps(wav, model, sampling_rate=SR, return_seconds=True, **vad_params)
        done[r["key"]] = float(sum(t["end"] - t["start"] for t in ts))
    save_json(done, out_path)
    return done


def data_statistics(manifest, speech):
    """The 'to measure' table: n, recordings, speakers by gender, durations, word statistics (M1)."""
    by_lang = defaultdict(list)
    for r in manifest["paired"]:
        by_lang[r["lang"]].append(r)
    stats = {"n_paired_indices": len(manifest["paired_ids"]),
             "excluded_too_long": manifest["excluded_too_long"],
             "n_number_indices": len(manifest["number_ids"])}
    for lang, recs in by_lang.items():
        dur = np.array([r["dur"] for r in recs])
        sp = np.array([speech.get(r["key"], np.nan) for r in recs])
        words = np.array([len(norm_basic(r["ref"]).split()) for r in recs])
        chars = np.array([len(norm_basic(r["ref"]).replace(" ", "")) for r in recs])
        per_idx = defaultdict(int)
        for r in recs:
            per_idx[r["idx"]] += 1
        vals, counts = np.unique(list(per_idx.values()), return_counts=True)
        stats[lang] = {
            "recordings": len(recs),
            "recordings_per_index": {str(int(v)): int(c) for v, c in zip(vals, counts)},
            "gender": {g: sum(r["gender"] == g for r in recs) for g in sorted({r["gender"] for r in recs})},
            "hours_total": float(dur.sum() / 3600),
            "hours_speech": float(np.nansum(sp) / 3600),
            "dur_median": float(np.median(dur)), "dur_p90": float(np.percentile(dur, 90)),
            "dur_max": float(dur.max()), "share_over_15s": float((dur > 15).mean()),
            "chars_per_word": float(chars.sum() / words.sum()),
            "sec_per_word_total": float(dur.sum() / words.sum()),
            "sec_per_word_speech": float(np.nansum(sp) / words.sum()),
            "chars_per_speech_sec": float(chars.sum() / np.nansum(sp)),
        }
    return stats


def index_features(manifest, speech):
    """Per sentence index and language: pooled speech seconds per word (M3 predictor)."""
    acc = defaultdict(lambda: [0.0, 0])
    for r in manifest["paired"]:
        k = (r["lang"], r["idx"])
        acc[k][0] += speech.get(r["key"], np.nan)
        acc[k][1] += len(norm_basic(r["ref"]).split())
    return {f"{l}:{i}": v[0] / v[1] for (l, i), v in acc.items() if v[1] > 0}

In [ ]:
%%writefile /kaggle/working/wspb/commit.py
"""Commit rules of the three re-decoding policies.

Each step receives the hypothesis of the whole buffer (re-decoded from its start with an empty
prompt) as a list of (start, end, word). The new-tail step is Whisper-Streaming's
HypothesisBuffer.insert (ufal/whisper_streaming, commit 6da90b4, whisper_online.py): keep the words
starting after the last committed word's end minus 0.1 s, then drop up to 5 words that repeat the
end of the committed text. The policies differ only in what they commit from that tail.

Because the hypothesis does not depend on what was committed (no prompt, no buffer trimming),
all three policies can be evaluated on the same sequence of decodes.
"""


class _TailState:
    def __init__(self):
        self.committed_in_buffer = []   # committed (start, end, word); never trimmed (trimming off)
        self.last_committed_time = 0.0

    def insert(self, hyp):
        new = [(a, b, t) for a, b, t in hyp if a > self.last_committed_time - 0.1]
        if len(new) >= 1:
            a, b, t = new[0]
            if abs(a - self.last_committed_time) < 1 and self.committed_in_buffer:
                cn, nn = len(self.committed_in_buffer), len(new)
                for i in range(1, min(min(cn, nn), 5) + 1):
                    c = " ".join([self.committed_in_buffer[-j][2] for j in range(1, i + 1)][::-1])
                    tail = " ".join(new[j - 1][2] for j in range(1, i + 1))
                    if c == tail:
                        for _ in range(i):
                            new.pop(0)
                        break
        return new

    def committed(self, words):
        if words:
            self.committed_in_buffer.extend(words)
            self.last_committed_time = words[-1][1]


class LocalAgreement:
    """LocalAgreement-2: commit the longest common prefix of the new tail and the previous tail
    (HypothesisBuffer.flush). The displayed tail is the unconfirmed rest (complete())."""

    name = "la2"

    def __init__(self):
        self.s = _TailState()
        self.buffer = []

    def step(self, hyp):
        new = self.s.insert(hyp)
        commit = []
        while new:
            if len(self.buffer) == 0:
                break
            if new[0][2] == self.buffer[0][2]:
                commit.append(new.pop(0))
                self.buffer.pop(0)
            else:
                break
        self.buffer = new
        self.s.committed(commit)
        return commit, list(self.buffer)

    def finish(self):
        c, self.buffer = self.buffer, []
        self.s.committed(c)
        return c


class HoldN:
    """Hold-n: commit every new word except the last n; hold-0 is the fixed-chunk policy."""

    def __init__(self, n):
        self.n = n
        self.name = f"hold{n}"
        self.s = _TailState()
        self.tail = []

    def step(self, hyp):
        new = self.s.insert(hyp)
        k = max(0, len(new) - self.n)
        commit, self.tail = new[:k], new[k:]
        self.s.committed(commit)
        return commit, list(self.tail)

    def finish(self):
        c, self.tail = self.tail, []
        self.s.committed(c)
        return c


def make_redecode_policies(hold_n):
    return [HoldN(0), HoldN(hold_n), LocalAgreement()]


def self_test():
    """Scripted hypothesis sequences with hand-computed commits (run in the notebook)."""
    W = lambda *ws: [(i * 0.5, i * 0.5 + 0.4, w) for i, w in enumerate(ws)]

    la = LocalAgreement()
    assert la.step(W("a", "b")) == ([], [(0, .4, "a"), (.5, .9, "b")])
    c, t = la.step(W("a", "b", "x"))
    assert [w[2] for w in c] == ["a", "b"] and [w[2] for w in t] == ["x"]
    c, t = la.step(W("a", "b", "c", "d"))           # "x" changed to "c": nothing agrees
    assert c == [] and [w[2] for w in t] == ["c", "d"]
    c, t = la.step(W("a", "b", "c", "d", "e"))
    assert [w[2] for w in c] == ["c", "d"] and [w[2] for w in t] == ["e"]
    assert [w[2] for w in la.finish()] == ["e"]

    h2 = HoldN(2)
    c, t = h2.step(W("a", "b", "c"))
    assert [w[2] for w in c] == ["a"] and [w[2] for w in t] == ["b", "c"]
    c, t = h2.step(W("a", "B", "c", "d"))           # committed "a" is never revised
    assert [w[2] for w in c] == ["B"] and [w[2] for w in t] == ["c", "d"], (c, t)
    h0 = HoldN(0)
    c, t = h0.step(W("a", "b"))
    assert [w[2] for w in c] == ["a", "b"] and t == []
    assert h0.finish() == []

    # n-gram de-duplication: a re-decoded word starting before the last committed end - 0.1 s is dropped
    la = LocalAgreement()
    la.step([(0.0, 0.4, "a"), (0.5, 0.9, "b")])
    la.step([(0.0, 0.4, "a"), (0.5, 0.9, "b"), (1.0, 1.4, "c")])   # commits a, b
    c, t = la.step([(0.0, 0.4, "a"), (0.85, 1.0, "b"), (1.0, 1.4, "c"), (1.5, 1.9, "d")])
    # "b" at 0.85 > 0.9-0.1 survives the time filter and repeats the committed "b": dropped
    assert [w[2] for w in c] == ["c"] and [w[2] for w in t] == ["d"], (c, t)
    return True

In [ ]:
%%writefile /kaggle/working/wspb/engines.py
"""Model wrappers. All harness engines load the same large-v3 checkpoint file, in fp32 as
SimulStreaming ships, with greedy decoding and a fixed language. AlignAtt runs through
SimulStreaming's own code; the re-decoding policies and the offline runs use the openai-whisper
package (SimulStreaming's bundled transcribe() predates large-v3: it always computes 80 mel bands,
while large-v3 needs 128)."""
import os

import numpy as np

from .common import SR


def _cuda():
    import torch
    return torch.cuda.is_available()


class RedecodeEngine:
    """Re-decodes the whole buffer from its start with an empty prompt and returns word timestamps
    (the input of hold-0, hold-n and LocalAgreement-2). Mirrors Whisper-Streaming's
    whisper_timestamped backend: greedy (beam 5 for the sensitivity runs), no temperature fallback, word timestamps,
    condition_on_previous_text=True (it only acts across 30 s windows, which never occur here)."""

    def __init__(self, model_name, lang, model_dir, beam=1):
        from whisper import load_model
        from whisper.transcribe import transcribe
        self.model = load_model(model_name, download_root=model_dir)
        self.transcribe = transcribe
        self.lang = lang
        # beam > 1: beam search with a single temperature (no fallback), so every step is still one
        # decode; used only for the LocalAgreement sensitivity runs (review R3)
        self.beam = beam
        self.decode(np.zeros(SR, dtype=np.float32))  # warm-up

    def decode(self, audio):
        kw = dict(beam_size=self.beam) if self.beam > 1 else {}
        r = self.transcribe(self.model, audio, language=self.lang, task="transcribe",
                            temperature=0.0, word_timestamps=True, condition_on_previous_text=True,
                            initial_prompt=None, fp16=False, verbose=None, **kw)
        words = []
        for seg in r["segments"]:
            for w in seg.get("words", []):
                t = w["word"].strip()
                if t:
                    words.append((float(w["start"]), float(w["end"]), t))
        return words

    def offline_text(self, audio, beam=False):
        kw = dict(language=self.lang, task="transcribe", fp16=False, verbose=None,
                  condition_on_previous_text=True)
        if beam:
            # Decoding of OpenAI's FLEURS notebook (beam 5, best_of 5, temperature fallback)
            kw.update(beam_size=5, best_of=5, temperature=(0.0, 0.2, 0.4, 0.6, 0.8, 1.0))
        else:
            kw.update(temperature=0.0)
        return self.transcribe(self.model, audio, **kw)["text"]

    def close(self):
        del self.model
        _free()


class AlignAttEngine:
    """SimulStreaming's own SimulWhisperASR + SimulWhisperOnline, unchanged, with settings as the
    command line sets them (frame threshold 25, audio_max_len 30, audio_min_len 0, greedy,
    no CIF model, no prompt, no logdir). `trim` sets --never_fire:
      trim=False (default as released): the last decoded word is kept;
      trim=True  (--never_fire): the last decoded word is always held back.
    Two read-only wrappers record what infer() already computes: the held-back word
    (generation['result_truncated']) and the alignment-head attention (for attention spread)."""

    def __init__(self, model_name, lang, model_dir, chunk, frame_threshold=25, trim=False):
        from simulstreaming_whisper import SimulWhisperASR, SimulWhisperOnline
        self.asr = SimulWhisperASR(
            language=lang, model_path=os.path.join(model_dir, model_name + ".pt"),
            cif_ckpt_path=None, frame_threshold=frame_threshold, audio_max_len=30.0,
            audio_min_len=0.0, segment_length=chunk, beams=1, task="transcribe",
            decoder_type="greedy", never_fire=trim, init_prompt=None, static_init_prompt=None,
            max_context_tokens=None, logdir=None)
        self.online = SimulWhisperOnline(self.asr)
        m = self.asr.model
        self._m = m
        self._attns = None
        self.last_gen = None
        orig_clean, orig_infer = m._clean_cache, m.infer

        def clean():
            self._attns = m.dec_attns          # keep the list the original is about to drop
            orig_clean()

        def infer(is_last=False):
            self._attns = None
            toks, gen = orig_infer(is_last=is_last)
            self.last_gen = gen
            return toks, gen

        m._clean_cache, m.infer = clean, infer
        self.asr.warmup(np.zeros(SR, dtype=np.float32))

    def attention_spread(self):
        """Mean entropy (nats) of the alignment heads' attention over the received frames, for the
        query position of every decoding pass of the last infer() call."""
        import torch
        gen, attns = self.last_gen, self._attns
        if not gen or not attns:
            return None
        L = len(self._m.model.decoder.blocks)
        T = int(gen["frames_len"])
        if T <= 1:
            return None
        ents = []
        for p in range(len(attns) // L):
            for layer, heads in self._m.align_source.items():
                a = attns[p * L + layer]                     # heads x queries x 1500
                for _, h in heads:
                    q = a[h, -1, :T].float()
                    q = q / q.sum().clamp_min(1e-12)
                    ents.append(float(-(q * torch.log(q.clamp_min(1e-12))).sum()))
        return float(np.mean(ents)) if ents else None

    def start(self):
        self.online.init()

    def _out(self, o):
        # The committed piece is the decoded text of the new tokens, as SimulStreaming outputs it:
        # pieces are joined by plain concatenation, so a word split across chunks ("pemot" +
        # "ongan") is rebuilt exactly as the tool's own output would show it.
        piece = o.get("text", "") if o else ""
        gen = self.last_gen or {}
        rt = gen.get("result_truncated") if isinstance(gen, dict) else None
        tail = "".join(rt["split_words"]) if rt else ""
        aux = {"spread": self.attention_spread(),
               "frames": gen.get("frames_len") if isinstance(gen, dict) else None}
        return piece, tail, aux

    def step(self, new_audio):
        self.online.insert_audio_chunk(new_audio)
        return self._out(self.online.process_iter())

    def finish(self):
        return self._out(self.online.finish())

    def close(self):
        del self.online, self.asr, self._m
        _free()


class AnchorEngine:
    """Whisper-Streaming as released (commit 6da90b4): FasterWhisperASR (beam 5, as hard-coded) +
    OnlineASRProcessor with its default segment trimming at 15 s; model large-v3, language fixed."""

    def __init__(self, lang, ws_dir, model_size="large-v3", cache_dir=None):
        import sys
        if ws_dir not in sys.path:
            sys.path.insert(0, ws_dir)
        from whisper_online import FasterWhisperASR, OnlineASRProcessor
        self.asr = FasterWhisperASR(lang, model_size, cache_dir=cache_dir)
        self.online = OnlineASRProcessor(self.asr)
        self.asr.transcribe(np.zeros(SR, dtype=np.float32))  # warm-up

    def start(self):
        self.online.init()

    def _out(self, o):
        # faster-whisper words carry their leading space and the tool joins them with sep=""
        tail = "".join(w for _, _, w in self.online.transcript_buffer.complete())
        return o[2] or "", tail, {}

    def step(self, new_audio):
        self.online.insert_audio_chunk(new_audio)
        return self._out(self.online.process_iter())

    def finish(self):
        o = self.online.finish()
        return o[2] or "", "", {}

    def close(self):
        del self.online, self.asr
        _free()


def _free():
    import gc
    gc.collect()
    if _cuda():
        import torch
        torch.cuda.empty_cache()

In [ ]:
%%writefile /kaggle/working/wspb/sim.py
"""Streaming simulation of one recording.

Computation-unaware (CU), as SimulStreaming's --comp_unaware loop: audio is processed at
end = chunk, 2*chunk, ..., duration; a word committed at a step is emitted at `end`.

Computation-aware (CA), as the real-time loop of whisper_online_main.py, on a virtual clock
(no sleeping): a step starts at max(clock, previous end + chunk), takes all audio received by
then, and emits its words when its measured compute time has passed. When processing falls
behind, steps take longer pieces of audio, exactly as in a live run.

When no CA step started later than its schedule, the CA steps process the same audio pieces as
the CU steps, so the CU record is derived from the CA run (same outputs, emission at `end`)
instead of being decoded again.

Record layout (per mode): steps = [[end, emit, compute], ...]; per policy: per step
[committed_piece, displayed_tail] as raw text. Pieces are joined by plain concatenation (each
re-decoding word is written with its leading space), so the transcript is "".join(pieces).
The last step entry of every policy is the finish step.
"""
import math

from .commit import make_redecode_policies


def _text(words):
    return "".join(" " + w for _, _, w in words)
from .common import SR, Timer

EPS = 1e-9


def _schedule_step(mode, prev_end, clock, chunk, dur):
    if mode == "cu":
        end = min(prev_end + chunk, dur)
        return end, end
    start = max(clock, prev_end + chunk)
    return start, min(start, dur)


def _run_redecode(engine, audio, dur, chunk, mode, hold_n, cuda):
    pols = make_redecode_policies(hold_n)
    out = {p.name: [] for p in pols}
    steps, prev_end, clock, lag = [], 0.0, 0.0, False
    while True:
        start, end = _schedule_step(mode, prev_end, clock, chunk, dur)
        if mode == "ca" and start > prev_end + chunk + EPS:
            lag = True
        with Timer(cuda) as t:
            hyp = engine.decode(audio[: int(round(end * SR))])
        clock = start + t.elapsed
        steps.append([end, end if mode == "cu" else clock, t.elapsed])
        for p in pols:
            c, tail = p.step(hyp)
            out[p.name].append([_text(c), _text(tail)])
        prev_end = end
        if end >= dur - EPS:
            break
    # finish: the policies commit their remaining tail without another decode (as Whisper-Streaming)
    steps.append([dur, dur if mode == "cu" else clock, 0.0])
    for p in pols:
        out[p.name].append([_text(p.finish()), ""])
    return {"steps": steps, "policies": out, "lag": lag}


def _run_incremental(engine, audio, dur, chunk, mode, cuda, name):
    """AlignAtt (and the anchor): new audio pieces are inserted into the engine's own buffer."""
    engine.start()
    steps, recs, aux = [], [], []
    prev_end, clock, lag = 0.0, 0.0, False
    while True:
        start, end = _schedule_step(mode, prev_end, clock, chunk, dur)
        if mode == "ca" and start > prev_end + chunk + EPS:
            lag = True
        piece = audio[int(round(prev_end * SR)): int(round(end * SR))]
        with Timer(cuda) as t:
            text, tail, a = engine.step(piece)
        clock = start + t.elapsed
        steps.append([end, end if mode == "cu" else clock, t.elapsed])
        recs.append([text, tail])
        aux.append(a)
        prev_end = end
        if end >= dur - EPS:
            break
    with Timer(cuda) as t:
        text, tail, a = engine.finish()
    clock += t.elapsed
    steps.append([dur, dur if mode == "cu" else clock, t.elapsed])
    recs.append([text, tail])
    aux.append(a)
    return {"steps": steps, "policies": {name: recs}, "aux": aux, "lag": lag}


def simulate(kind, engine, audio, chunk, hold_n=2, policy_name=None, cuda=True, modes=("ca", "cu")):
    """modes=("ca", "cu"): run CA; run CU separately only if CA fell behind its schedule.
    modes=("cu",) or ("ca",): run only that mode; the other is None."""
    dur = len(audio) / SR
    run = (lambda mode: _run_redecode(engine, audio, dur, chunk, mode, hold_n, cuda)) \
        if kind == "redecode" else \
        (lambda mode: _run_incremental(engine, audio, dur, chunk, mode, cuda, policy_name))
    if "ca" not in modes:
        cu = run("cu")
        cu["derived"] = False
        cu["n_expected"] = int(math.ceil(dur / chunk - EPS)) + 1
        return {"dur": dur, "ca": None, "cu": cu}
    ca = run("ca")
    if "cu" not in modes:
        return {"dur": dur, "ca": ca, "cu": None}
    if ca["lag"]:
        cu = run("cu")
        cu["derived"] = False
    else:
        cu = {k: v for k, v in ca.items()}
        cu["steps"] = [[end, end if i < len(ca["steps"]) - 1 else dur, c]
                       for i, (end, _, c) in enumerate(ca["steps"])]
        cu["derived"] = True
    # number of CU steps expected without lag, for a consistency check
    cu["n_expected"] = int(math.ceil(dur / chunk - EPS)) + 1
    return {"dur": dur, "ca": ca, "cu": cu}

In [ ]:
%%writefile /kaggle/working/wspb/worker.py
"""Job runner. Usage: python -m wspb.worker --config CONFIG --worker N

Each worker takes the next unfinished job in priority order (claimed with an exclusive lock
file), runs it one recording at a time, appends one JSON line per recording, and stops starting
new recordings after the session deadline. A job interrupted by the deadline continues in the
next session from its last saved recording.
"""
import argparse
import os
import time
import traceback
from pathlib import Path

from .common import add_simulstreaming_to_path, append_jsonl, load_config, log, read_jsonl


def job_paths(cfg, job_id):
    rd = Path(cfg["results_dir"])
    return rd / f"{job_id}.jsonl", rd / f"{job_id}.done"


def records_for(job, manifest, quick_ids):
    recs = manifest["full"] if job.get("set") == "full" else manifest["paired"]
    recs = [r for r in recs if r["lang"] == job["lang"]]
    if job.get("ids") is not None:          # CA timing subset
        keep = set(job["ids"])
        recs = [r for r in recs if r["idx"] in keep]
    if quick_ids is not None:
        recs = [r for r in recs if r["idx"] in quick_ids]
    return recs


def claim(cfg, job_id):
    d = Path(cfg["results_dir"]) / "claims"
    d.mkdir(parents=True, exist_ok=True)
    try:
        fd = os.open(d / f"{job_id}.lock", os.O_CREAT | os.O_EXCL | os.O_WRONLY)
        os.close(fd)
        return True
    except FileExistsError:
        return False


def build_engine(job, cfg):
    from . import engines
    lang = job["lang"]
    if job["kind"] == "redecode":
        return engines.RedecodeEngine(job["model"], lang, cfg["model_dir"], beam=job.get("beam", 1))
    if job["kind"] == "offline":
        return engines.RedecodeEngine(job["model"], lang, cfg["model_dir"])
    if job["kind"] == "aa":
        return engines.AlignAttEngine(job["model"], lang, cfg["model_dir"], job["chunk"],
                                      frame_threshold=cfg["frame_threshold"], trim=job["trim"])
    if job["kind"] == "anchor":
        return engines.AnchorEngine(lang, cfg["whisper_streaming_dir"], cache_dir=cfg["model_dir"])
    raise ValueError(job["kind"])


def run_job(job, cfg, manifest, deadline, logfile):
    import torch
    from .common import Timer
    from .data import load_audio
    from .sim import simulate

    cuda = torch.cuda.is_available()
    out, done = job_paths(cfg, job["id"])
    finished = {r["key"] for r in read_jsonl(out)}
    todo = [r for r in records_for(job, manifest, cfg.get("quick_ids")) if r["key"] not in finished]
    log(f"{job['id']}: {len(todo)} recordings to run ({len(finished)} already saved)", logfile)
    if not todo:
        done.touch()
        return True
    engine = build_engine(job, cfg)
    t_job = time.time()
    try:
        for i, r in enumerate(todo):
            if time.time() > deadline:
                log(f"{job['id']}: deadline reached after {i} recordings; resume next session", logfile)
                return False
            t_rec = time.time()
            audio = load_audio(r["path"])
            line = {"key": r["key"], "idx": r["idx"], "lang": r["lang"], "dur": len(audio) / 16000}
            if job["kind"] == "offline":
                with Timer(cuda) as t:
                    line["text"] = engine.offline_text(audio, beam=job.get("beam", False))
                line["compute"] = t.elapsed
            else:
                name = job.get("policy_name")
                res = simulate(job["kind"] if job["kind"] == "redecode" else "incremental",
                               engine, audio, job["chunk"], hold_n=cfg["hold_n"],
                               policy_name=name, cuda=cuda,
                               modes=tuple(job.get("modes", ("ca", "cu"))))
                line.update(res)
            line["wall"] = time.time() - t_rec
            append_jsonl(line, out)
            if (i + 1) % 25 == 0:
                rate = (time.time() - t_job) / (i + 1)
                log(f"{job['id']}: {i + 1}/{len(todo)}  ~{rate * (len(todo) - i - 1) / 3600:.2f} h left", logfile)
    finally:
        engine.close()
    done.touch()
    log(f"{job['id']}: done in {(time.time() - t_job) / 3600:.2f} h", logfile)
    return True


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--worker", type=int, default=0)
    a = ap.parse_args()
    cfg = load_config(a.config)
    add_simulstreaming_to_path(cfg)
    from .data import load_manifest
    manifest = load_manifest(cfg["data_dir"])
    if cfg.get("quick_ids") is not None:
        cfg["quick_ids"] = set(cfg["quick_ids"])
    logfile = Path(cfg["results_dir"]) / "logs" / f"worker{a.worker}.log"
    logfile.parent.mkdir(parents=True, exist_ok=True)
    deadline = cfg["deadline_epoch"]
    log(f"worker {a.worker} started; GPU {os.environ.get('CUDA_VISIBLE_DEVICES')}", logfile)
    for job in cfg["jobs"]:
        if time.time() > deadline:
            break
        _, done = job_paths(cfg, job["id"])
        if done.exists() or not claim(cfg, job["id"]):
            continue
        try:
            run_job(job, cfg, manifest, deadline, logfile)
        except Exception:
            log(f"{job['id']}: FAILED\n{traceback.format_exc()}", logfile)
    log(f"worker {a.worker} finished", logfile)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/wspb/metrics.py
"""Per-recording metrics from the saved simulation records.

Transcript: committed pieces joined by plain concatenation. A word's delay is the time at which
its last character was committed (so a word split across chunks counts when it is complete).

WER          Whisper normaliser per language, word Levenshtein alignment.
CER          same normaliser, spaces removed (insensitive to word segmentation; review R5).
split/merge  word-segmentation errors in the WER alignment ("dirumah" vs "di rumah").
LAAL         Papi et al. (2022), on raw output words; tau from the source time consumed (step end).
             CU: delay = step end (finish = duration). CA: delay = emission on the virtual clock.
first output delay of the first output word (CU and CA).
RTF          total compute time / duration (CU record).
erasure      Arivazhagan et al. (2020) on the display = committed text + uncommitted tail:
             sum over updates of (|previous display| - common prefix) / |final transcript|;
             primary after BasicTextNormalizer, secondary on the raw display.
tail conf.   share of displayed tail words later committed unchanged at the same position.
M4 (E)       share of committed (hypothesis-side) words that are substituted or inserted.
"""
import re

import numpy as np

from .align import align, counts
from .text import cer_counts, norm_basic, split_merge, words_wer

_WORD = re.compile(r"\S+")


def word_delays(pieces, emits, ends):
    full = "".join(pieces)
    step_of_char = np.empty(len(full), dtype=np.int32)
    pos = 0
    for k, p in enumerate(pieces):
        step_of_char[pos:pos + len(p)] = k
        pos += len(p)
    words, d_emit, d_src = [], [], []
    for m in _WORD.finditer(full):
        k = int(step_of_char[m.end() - 1])
        words.append(m.group())
        d_emit.append(emits[k])
        d_src.append(ends[k])
    return full.strip(), words, np.array(d_emit), np.array(d_src)


def laal(delays, src, dur, n_hyp, n_ref):
    if n_hyp == 0:
        return np.nan
    reach = np.nonzero(src >= dur - 1e-6)[0]
    tau = int(reach[0]) + 1 if len(reach) else n_hyp
    rate = dur / max(n_hyp, n_ref)
    i = np.arange(tau)
    return float(np.mean(delays[:tau] - i * rate))


def _lcp(a, b):
    n = min(len(a), len(b))
    i = 0
    while i < n and a[i] == b[i]:
        i += 1
    return i


def display_metrics(pieces, tails, final_text):
    final_n = norm_basic(final_text).split()
    final_r = final_text.split()
    prev_n, prev_r = [], []
    er_n = er_r = 0
    conf = total = 0
    committed = ""
    for piece, tail in zip(pieces, tails):
        committed += piece
        disp_n = norm_basic(committed + tail).split()
        disp_r = (committed + tail).split()
        er_n += len(prev_n) - _lcp(prev_n, disp_n)
        er_r += len(prev_r) - _lcp(prev_r, disp_r)
        prev_n, prev_r = disp_n, disp_r
        if tail.strip():
            start = len(norm_basic(committed).split())
            for j, w in enumerate(norm_basic(tail).split()):
                total += 1
                conf += int(start + j < len(final_n) and final_n[start + j] == w)
    return (er_n / max(1, len(final_n)), er_r / max(1, len(final_r)), conf, total)


def recording_metrics(rec, policy, mode_rec, ref, lang):
    steps = mode_rec["steps"]
    emits = [s[1] for s in steps]
    ends = [s[0] for s in steps]
    pieces = [p for p, _ in mode_rec["policies"][policy]]
    tails = [t for _, t in mode_rec["policies"][policy]]
    final, words, d_emit, d_src = word_delays(pieces, emits, ends)
    ref_w, hyp_w = words_wer(ref, lang), words_wer(final, lang)
    ops = align(ref_w, hyp_w)
    c = counts(ops)
    sp, mg = split_merge(ops, ref_w, hyp_w)
    ce, cn = cer_counts(ref, final, lang)
    n_ref_raw = len(ref.split())
    er_n, er_r, conf, total = display_metrics(pieces, tails, final)
    dur = rec["dur"]
    return {
        "S": c["S"], "D": c["D"], "I": c["I"], "C": c["C"], "N": len(ref_w), "H": len(hyp_w),
        "CE": ce, "CN": cn, "splits": sp, "merges": mg,
        "laal": laal(d_emit, d_src, dur, len(words), n_ref_raw),
        "first": float(d_emit[0]) if len(words) else np.nan,
        "compute": float(sum(s[2] for s in steps)),
        "erasure": er_n, "erasure_raw": er_r, "tail_conf": conf, "tail_total": total,
        "text": final,
    }


def offline_metrics(line, ref, lang):
    ref_w, hyp_w = words_wer(ref, lang), words_wer(line["text"], lang)
    ops = align(ref_w, hyp_w)
    c = counts(ops)
    sp, mg = split_merge(ops, ref_w, hyp_w)
    ce, cn = cer_counts(ref, line["text"], lang)
    return {"S": c["S"], "D": c["D"], "I": c["I"], "C": c["C"], "N": len(ref_w), "H": len(hyp_w),
            "CE": ce, "CN": cn, "splits": sp, "merges": mg,
            "compute": line.get("compute", np.nan), "text": line["text"].strip()}


def collect(results_dir, jobs, manifest):
    """One row per (job, policy, mode, recording)."""
    import pandas as pd
    from .common import read_jsonl

    from collections import Counter
    refs = {r["key"]: r for r in manifest["full"]}
    # recordings per (language, sentence index) in the paired set, for the equal-weight sensitivity
    n_rec = Counter((r["lang"], r["idx"]) for r in manifest["paired"])
    rows = []
    for job in jobs:
        lines = read_jsonl(f"{results_dir}/{job['id']}.jsonl")
        for line in lines:
            r = refs[line["key"]]
            base = {"job": job["id"], "kind": job["kind"], "lang": line["lang"], "idx": line["idx"],
                    "key": line["key"], "dur": line["dur"], "chunk": job.get("chunk", np.nan)}
            if job["kind"] == "offline":
                rows.append({**base, "policy": job["policy_name"], "mode": "offline",
                             "n_rec_idx": n_rec.get((line["lang"], line["idx"]), 1),
                             **offline_metrics(line, r["ref"], r["lang"])})
                continue
            suffix = job.get("policy_suffix", "")
            # CA timing jobs (subset of indices) contribute latency rows only: their CU records, if
            # any, would duplicate the main jobs' text results for those indices
            modes = ("ca",) if job.get("ca_timing") else ("cu", "ca")
            ca_line = line.get("ca") or {}
            for mode in modes:
                mrec = line.get(mode)
                if not mrec:
                    continue
                for pol in mrec["policies"]:
                    m = recording_metrics(line, pol, mrec, r["ref"], r["lang"])
                    m["n_rec_idx"] = n_rec[(line["lang"], line["idx"])]
                    if mode == "cu" and pol.startswith("aa"):
                        sp = [a["spread"] for a in mrec.get("aux", []) if a and a.get("spread") is not None]
                        m["spread"] = float(np.mean(sp)) if sp else np.nan
                    m["lag"] = bool(ca_line["lag"]) if ca_line else np.nan
                    m["cu_derived"] = bool((line.get("cu") or {}).get("derived", False))
                    rows.append({**base, "policy": pol + suffix, "mode": mode, **m})
    return pd.DataFrame(rows)

In [ ]:
%%writefile /kaggle/working/wspb/stats.py
"""Planned analyses (fixed before the runs). Primary text metrics use the computation-unaware (CU) records,
whose output depends only on the audio and the chunk size, not on the GPU.

Bootstrap: sentence indices resampled with replacement, the same indices for both languages
(paired), B resamples (10,000 by default), fixed seed. Two-sided bootstrap p-value:
2 * min(P(stat* <= 0), P(stat* >= 0)). Holm correction within each family.

H3 estimand (fixed before the runs, review R2). For a language L and chunk size c,
    R_L = (E_LA - E_AA) / E_LA,
where E_P is the pooled error count of policy P over all recordings of the paired indices
(a ratio of sums, never a mean of per-sentence ratios, so sentences with zero errors need no
special handling; N is the same for both policies, so the ratio of WERs is the same number).
R is recomputed inside every bootstrap resample; a resample with E_LA = 0 is dropped (reported).
Delta = R_ID - R_EN. Co-reported: the absolute gap
    A = (WER_LA,ID - WER_AA,ID) - (WER_LA,EN - WER_AA,EN)   (WER points).
Equivalence ("ruled out"): the 95% CI of Delta lies entirely inside +/- equivalence_margin
(default 0.42 = the 7%-49% spread of Simul-Whisper's per-language results). The smallest change
the data exclude, max(|lo|, |hi|), is always reported.

H3 at matched latency (fixed before the runs, review R1). For each language and H3 chunk size c,
AlignAtt's WER is linearly interpolated along AlignAtt's own WER-LAAL curve (its chunk sizes
0.5-2.0 s) at LocalAgreement's LAAL at c (primary); the reverse anchoring is secondary. No
extrapolation. Latency axis: computation-unaware LAAL (primary; it does not depend on the GPU),
computation-aware LAAL (secondary). R*, Delta* as above,
recomputed per resample. Tie-break: when the fixed-chunk and matched-latency verdicts differ, both
are reported and the abstract uses the weaker claim ("bounds"/"suggests", never "shows").
"""
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

Z_MDE = 2.241 + 0.842   # z(1 - 0.025/2) + z(0.80)


def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    m = len(p)
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj.tolist()


def boot_p(dist):
    dist = np.asarray(dist)
    dist = dist[np.isfinite(dist)]
    if len(dist) == 0:
        return np.nan
    return float(min(1.0, 2 * min((dist <= 0).mean(), (dist >= 0).mean())))


def ci(dist):
    dist = np.asarray(dist)
    dist = dist[np.isfinite(dist)]
    return [float(np.percentile(dist, 2.5)), float(np.percentile(dist, 97.5))] if len(dist) else [np.nan, np.nan]


class Data:
    """Per-index pooled error counts for every (policy, lang, chunk) in the CU records."""

    def __init__(self, df, ids, B=10000, seed=20261022):
        self.df = df
        self.ids = list(ids)
        self.n = len(self.ids)
        rng = np.random.default_rng(seed)
        self.W = rng.multinomial(self.n, np.full(self.n, 1 / self.n), size=B).astype(np.float64)
        self._cache = {}

    def counts(self, policy, lang, chunk=None, mode="cu", exclude=(), metric="wer", equal_weight=False):
        """Per-index (errors, reference units). metric: "wer" (words) or "cer" (characters).
        equal_weight: every sentence index weighs the same whatever its number of recordings
        (sensitivity analysis; the primary analysis pools recordings)."""
        key = (policy, lang, chunk, mode, tuple(sorted(exclude)), metric, equal_weight)
        if key in self._cache:
            return self._cache[key]
        d = self.df[(self.df.policy == policy) & (self.df.lang == lang) & (self.df["mode"] == mode)]
        if chunk is not None:
            d = d[np.isclose(d.chunk, chunk)]
        if metric == "cer":
            d = d.assign(E=d.CE.astype(float), N=d.CN.astype(float))
        else:
            d = d.assign(E=(d.S + d.D + d.I).astype(float), N=d.N.astype(float))
        if equal_weight:
            k = d.n_rec_idx.astype(float)
            d = d.assign(E=d.E / k, N=d.N / k)
        g = d.groupby("idx")[["E", "N"]].sum()
        g = g.reindex(self.ids)
        if g.isna().any().any():
            res = None                               # incomplete run: analysis skipped
        else:
            E, N = g.E.to_numpy(float), g.N.to_numpy(float)
            mask = np.array([i not in set(exclude) for i in self.ids], float)
            res = (E * mask, N * mask)
        self._cache[key] = res
        return res

    def wer(self, policy, lang, chunk=None, mode="cu", exclude=(), metric="wer", equal_weight=False):
        c = self.counts(policy, lang, chunk, mode, exclude, metric, equal_weight)
        if c is None:
            return None, None
        E, N = c
        return E.sum() / N.sum(), (self.W @ E) / (self.W @ N)

    def latency(self, policy, lang, chunk, mode="cu", col="laal"):
        """Mean per-index latency (recordings of an index averaged first) and its bootstrap."""
        key = ("lat", policy, lang, chunk, mode, col)
        if key in self._cache:
            return self._cache[key]
        x = per_index_mean(self.df, policy, lang, chunk, col, mode, self.ids)
        if x is None:
            res = (None, None)
        else:
            ok = np.isfinite(x)
            Wk = self.W[:, ok]
            res = (float(x[ok].mean()), (Wk @ x[ok]) / Wk.sum(1))
        self._cache[key] = res
        return res


def rel_adv(D, la, aa, lang, chunk, exclude=(), metric="wer", equal_weight=False):
    w_la, b_la = D.wer(la, lang, chunk, exclude=exclude, metric=metric, equal_weight=equal_weight)
    w_aa, b_aa = D.wer(aa, lang, chunk, exclude=exclude, metric=metric, equal_weight=equal_weight)
    if w_la is None or w_aa is None:
        return None, None
    with np.errstate(divide="ignore", invalid="ignore"):
        b = np.where(b_la > 0, (b_la - b_aa) / b_la, np.nan)
    return (w_la - w_aa) / w_la, b


def abs_adv(D, la, aa, lang, chunk, exclude=(), metric="wer", equal_weight=False):
    w_la, b_la = D.wer(la, lang, chunk, exclude=exclude, metric=metric, equal_weight=equal_weight)
    w_aa, b_aa = D.wer(aa, lang, chunk, exclude=exclude, metric=metric, equal_weight=equal_weight)
    if w_la is None or w_aa is None:
        return None, None
    return w_la - w_aa, b_la - b_aa


def equivalence(ci95, margin):
    lo, hi = ci95
    if not (np.isfinite(lo) and np.isfinite(hi)):
        return {"margin": margin, "verdict": "not available"}
    return {"margin": margin, "smallest_change_excluded": float(max(abs(lo), abs(hi))),
            "verdict": f"change larger than +/-{margin:.2f} ruled out" if -margin < lo and hi < margin
            else f"a change of +/-{margin:.2f} is not ruled out"}


def pratt(D, a, b, lang, chunk, metric="wer", equal_weight=False):
    """Wilcoxon signed-rank (Pratt) on per-index WER(a) - WER(b)."""
    ca = D.counts(a, lang, chunk, metric=metric, equal_weight=equal_weight)
    cb = D.counts(b, lang, chunk, metric=metric, equal_weight=equal_weight)
    if ca is None or cb is None:
        return None
    x = ca[0] / ca[1] - cb[0] / cb[1]
    if np.allclose(x, 0):
        return {"p": 1.0, "mean_diff": 0.0, "pooled_diff": 0.0}
    return {"p": float(wilcoxon(x, zero_method="pratt").pvalue), "mean_diff": float(x.mean()),
            "pooled_diff": float(ca[0].sum() / ca[1].sum() - cb[0].sum() / cb[1].sum())}


def verdict(p, est, pos="widens", neg="shrinks", null="no detectable change"):
    if p is None or not np.isfinite(p):
        return "not available"
    return (pos if est > 0 else neg) if p < 0.05 else null


def h3(D, aa, la="la2", chunks=(0.5, 1.0), exclude=(), metric="wer", equal_weight=False, margin=0.42):
    out = {"chunks": {}, "metric": metric, "equal_weight": equal_weight, "la": la, "aa": aa}
    pre, keep, h3p = [], [], []
    for c in chunks:
        pe = pratt(D, la, aa, "en", c, metric=metric, equal_weight=equal_weight)
        pi = pratt(D, la, aa, "id", c, metric=metric, equal_weight=equal_weight)
        r_en, b_en = rel_adv(D, la, aa, "en", c, exclude, metric, equal_weight)
        r_id, b_id = rel_adv(D, la, aa, "id", c, exclude, metric, equal_weight)
        if None in (pe, pi, r_en, r_id):
            out["chunks"][str(c)] = "not available"
            pre.append(np.nan); keep.append(np.nan); h3p.append(np.nan)
            continue
        delta, bd = r_id - r_en, b_id - b_en
        a_en, ba_en = abs_adv(D, la, aa, "en", c, exclude, metric, equal_weight)
        a_id, ba_id = abs_adv(D, la, aa, "id", c, exclude, metric, equal_weight)
        lat = {}
        for mode in ("cu", "ca"):
            for k, (p, l) in {"la_en": (la, "en"), "aa_en": (aa, "en"), "la_id": (la, "id"),
                              "aa_id": (aa, "id")}.items():
                lat[f"laal_{mode}_{k}"] = D.latency(p, l, c, mode)[0]
        ci_d = ci(bd)
        out["chunks"][str(c)] = {
            "R_en": r_en, "R_id": r_id, "delta": delta, "ci95_unadjusted": ci_d,
            "p_boot": boot_p(bd), "se": float(np.nanstd(bd)), "mde_80": float(Z_MDE * np.nanstd(bd)),
            "resamples_dropped": int((~np.isfinite(bd)).sum()),
            "equivalence": equivalence(ci_d, margin),
            "abs_gap_en": a_en, "abs_gap_id": a_id, "abs_delta": a_id - a_en,
            "abs_delta_ci95": ci(ba_id - ba_en), "abs_delta_p_boot": boot_p(ba_id - ba_en),
            "precondition_en": pe, "keeps_id": pi,
            "wer": {k: D.wer(p, l, c, exclude=exclude, metric=metric, equal_weight=equal_weight)[0]
                    for k, (p, l) in {"la_en": (la, "en"), "aa_en": (aa, "en"), "la_id": (la, "id"),
                                      "aa_id": (aa, "id")}.items()},
            "latency": lat,
        }
        pre.append(pe["p"]); keep.append(pi["p"]); h3p.append(boot_p(bd))
    # Holm within each family, over the chunk sizes whose p-value exists (a p-value can be
    # missing when too few indices make every bootstrap resample degenerate)
    for name, ps in (("precondition_holm", pre), ("keeps_holm", keep), ("h3_holm", h3p)):
        valid = [i for i, p in enumerate(ps) if np.isfinite(p)]
        if valid:
            for i, a in zip(valid, holm([ps[i] for i in valid])):
                out["chunks"][str(chunks[i])][name] = a
    for c in chunks:
        r = out["chunks"][str(c)]
        if not isinstance(r, dict):
            continue
        p_pre, p_keep = r.get("precondition_holm", np.nan), r.get("keeps_holm", np.nan)
        r["precondition_met"] = bool(p_pre < 0.05 and r["precondition_en"].get("pooled_diff", 0) > 0)
        r["keeps"] = bool(p_keep < 0.05 and r["keeps_id"].get("pooled_diff", 0) > 0)
        r["verdict"] = verdict(r.get("h3_holm", np.nan), r["delta"]) if r["precondition_met"] else \
            "not tested (precondition not met; descriptive only)"
    return out


def h4(D, policy, exclude=()):
    res = {}
    for lang in ("en", "id"):
        w1, b1 = D.wer(policy, lang, 1.0, exclude=exclude)
        w2, b2 = D.wer(policy, lang, 2.0, exclude=exclude)
        if w1 is None or w2 is None:
            return "not available"
        res[lang] = ((w1 - w2) / w2, (b1 - b2) / b2)
    d, bd = res["id"][0] - res["en"][0], res["id"][1] - res["en"][1]
    p = boot_p(bd)
    return {"P_en": res["en"][0], "P_id": res["id"][0], "diff": d, "ci95": ci(bd), "p_boot": p,
            "verdict": verdict(p, d, "larger penalty for Indonesian", "smaller penalty for Indonesian")}


def h2(D, policies, chunks):
    out = {}
    for pol in policies:
        for c in chunks:
            we, be = D.wer(pol, "en", c)
            wi, bi = D.wer(pol, "id", c)
            if we is None or wi is None:
                continue
            out[f"{pol}@{c}"] = {"wer_en": we, "wer_id": wi, "diff": wi - we, "ci95": ci(bi - be),
                                 "p_boot": boot_p(bi - be)}
    return out


def h1(D, policies, chunks):
    out = {}
    for lang in ("en", "id"):
        for c in chunks:
            w = {p: D.wer(p, lang, c)[0] for p in policies}
            if any(v is None for v in w.values()):
                continue
            out[f"{lang}@{c}"] = {"wer": w, "ranking": sorted(w, key=w.get)}
    return out


def per_index_mean(df, policy, lang, chunk, col, mode, ids):
    d = df[(df.policy == policy) & (df.lang == lang) & (df["mode"] == mode) & np.isclose(df.chunk, chunk)]
    s = d.groupby("idx")[col].mean().reindex(ids)
    return None if s.isna().all() else s.to_numpy(float)


def h5(D, aa, chunks, mode="cu"):
    """AlignAtt LAAL, Indonesian minus English, per chunk size. mode="cu" (primary, all indices) or
    "ca" (secondary, CA timing subset)."""
    out = {"mode": mode}
    for c in chunks:
        e = per_index_mean(D.df, aa, "en", c, "laal", mode, D.ids)
        i = per_index_mean(D.df, aa, "id", c, "laal", mode, D.ids)
        if e is None or i is None:
            continue
        x = i - e
        ok = np.isfinite(x)
        bd = (D.W[:, ok] @ x[ok]) / D.W[:, ok].sum(1)
        out[str(c)] = {"laal_en": float(np.nanmean(e)), "laal_id": float(np.nanmean(i)),
                       "n_indices": int(ok.sum()),
                       "diff": float(np.nanmean(x)), "ci95": ci(bd), "p_boot": boot_p(bd)}
    return out


def m3(D, sec_per_word, chunk=1.0, la="la2", aa="aa_keep"):
    """WLS per language: WER_LA - WER_AA (per index) ~ speech seconds per word + offline WER."""
    out = {}
    for lang in ("en", "id"):
        cl, ca_, co = D.counts(la, lang, chunk), D.counts(aa, lang, chunk), D.counts("offline_v3", lang, None, "offline")
        if None in (cl, ca_, co):
            continue
        y = cl[0] / cl[1] - ca_[0] / ca_[1]
        x = np.array([sec_per_word.get(f"{lang}:{i}", np.nan) for i in D.ids])
        z = co[0] / co[1]
        ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
        X = np.column_stack([np.ones(ok.sum()), x[ok], z[ok]])
        w = cl[1][ok]

        def fit(weights):
            XtW = X.T * weights
            return np.linalg.solve(XtW @ X, XtW @ y[ok])

        beta = fit(w)
        Wb = D.W[:, ok] * w
        XtWX = np.einsum("bn,ni,nj->bij", Wb, X, X)
        XtWy = np.einsum("bn,ni,n->bi", Wb, X, y[ok])
        slopes = np.array([np.linalg.lstsq(A, v, rcond=None)[0][1] for A, v in zip(XtWX, XtWy)])
        p = boot_p(slopes)
        out[lang] = {"slope_sec_per_word": float(beta[1]), "ci95": ci(slopes), "p_boot": p,
                     "verdict": verdict(p, beta[1], "supports M3", "contradicts M3", "does not support M3")}
    return out


def m4(D, aa_trim="aa_trim", chunks=(0.5, 1.0)):
    """X = T / E per language; T = share of held-back words that come back changed,
    E = share of committed words that are substituted or inserted."""
    out = {}
    df = D.df
    for c in chunks:
        res = {}
        for lang in ("en", "id"):
            d = df[(df.policy == aa_trim) & (df.lang == lang) & (df["mode"] == "cu") & np.isclose(df.chunk, c)]
            if d.empty:
                break
            g = d.assign(unconf=d.tail_total - d.tail_conf, err_h=d.S + d.I).groupby("idx")[
                ["unconf", "tail_total", "err_h", "H"]].sum().reindex(D.ids).fillna(0).to_numpy(float).T
            u, t, e, h = g
            X = (u.sum() / t.sum()) / (e.sum() / h.sum())
            Xb = ((D.W @ u) / (D.W @ t)) / ((D.W @ e) / (D.W @ h))
            res[lang] = (X, Xb, u.sum() / t.sum(), e.sum() / h.sum())
        if len(res) < 2:
            continue
        d, bd = res["id"][0] - res["en"][0], res["id"][1] - res["en"][1]
        lo, hi = ci(bd)
        v = "(b) supported" if lo > 0 else "(b) contradicted" if hi < 0 else \
            "(b) not supported; (a) remains the simpler reading, not a demonstrated one"
        out[str(c)] = {"X_en": res["en"][0], "X_id": res["id"][0], "T_en": res["en"][2], "T_id": res["id"][2],
                       "E_en": res["en"][3], "E_id": res["id"][3], "diff": d, "ci95": [lo, hi], "reading": v}
    return out


def m5_trim_gain(D, la="la2", chunks=(0.5, 1.0)):
    """Added before data (after finding that SimulStreaming keeps the last word by default):
    gain in relative advantage from holding back the last word, Indonesian minus English."""
    out = {}
    for c in chunks:
        g = {}
        for lang in ("en", "id"):
            rk, bk = rel_adv(D, la, "aa_keep", lang, c)
            rt, bt = rel_adv(D, la, "aa_trim", lang, c)
            if rk is None or rt is None:
                break
            g[lang] = (rt - rk, bt - bk)
        if len(g) < 2:
            continue
        d, bd = g["id"][0] - g["en"][0], g["id"][1] - g["en"][1]
        p = boot_p(bd)
        out[str(c)] = {"gain_en": g["en"][0], "gain_id": g["id"][0], "diff": d, "ci95": ci(bd),
                       "p_boot": p, "verdict": verdict(p, d, "trimming helps Indonesian more",
                                                       "trimming helps English more")}
    return out


def _interp_rows(x, xp, fp):
    """Per resample b: interpolate fp[:, b] against xp[:, b] at x[b]; NaN outside the range."""
    out = np.full(len(x), np.nan)
    for b in range(len(x)):
        ok = np.isfinite(xp[:, b]) & np.isfinite(fp[:, b])
        if ok.sum() < 2 or not np.isfinite(x[b]):
            continue
        o = np.argsort(xp[ok, b])
        out[b] = np.interp(x[b], xp[ok, b][o], fp[ok, b][o], left=np.nan, right=np.nan)
    return out


def h3_matched_latency(D, aa, la="la2", chunks=(0.5, 1.0), curve_chunks=(0.5, 1.0, 1.2, 1.5, 2.0),
                       lat_mode="cu", anchor="la", margin=0.42):
    """anchor="la" (primary): at LocalAgreement's LAAL at chunk c, AlignAtt's WER is interpolated
    along AlignAtt's own WER-LAAL curve. anchor="aa" (secondary): the reverse. No extrapolation."""
    fixed, curve = (la, aa) if anchor == "la" else (aa, la)
    out = {"latency_axis": "LAAL, " + ("computation-unaware" if lat_mode == "cu" else "computation-aware"),
           "anchor": fixed, "interpolated": curve, "chunks": {}}
    for c in chunks:
        res = {}
        for lang in ("en", "id"):
            pts = [(k, D.wer(curve, lang, k), D.latency(curve, lang, k, lat_mode)) for k in curve_chunks]
            pts = [(k, w, l) for k, w, l in pts if w[0] is not None and l[0] is not None]
            w_fx, b_fx = D.wer(fixed, lang, c)
            l_fx, bl_fx = D.latency(fixed, lang, c, lat_mode)
            if len(pts) < 2 or w_fx is None or l_fx is None:
                break
            xp = np.array([l[0] for _, _, l in pts])
            fp = np.array([w[0] for _, w, _ in pts])
            o = np.argsort(xp)
            w_cv = float(np.interp(l_fx, xp[o], fp[o], left=np.nan, right=np.nan))
            bw_cv = _interp_rows(bl_fx, np.stack([l[1] for _, _, l in pts]), np.stack([w[1] for _, w, _ in pts]))
            w_la, w_aa = (w_fx, w_cv) if anchor == "la" else (w_cv, w_fx)
            b_la, b_aa = (b_fx, bw_cv) if anchor == "la" else (bw_cv, b_fx)
            with np.errstate(divide="ignore", invalid="ignore"):
                r = (w_la - w_aa) / w_la if w_la > 0 else np.nan
                br = np.where(b_la > 0, (b_la - b_aa) / b_la, np.nan)
            res[lang] = {"laal_anchor": l_fx, "curve_laal_range": [float(xp.min()), float(xp.max())],
                         "curve_chunks_used": [k for k, _, _ in pts],
                         "wer_la": w_la, "wer_aa": w_aa, "R": r, "_b": br}
        if len(res) < 2:
            out["chunks"][str(c)] = "not available"
            continue
        d = res["id"]["R"] - res["en"]["R"]
        bd = res["id"].pop("_b") - res["en"].pop("_b")
        if not np.isfinite(d):
            out["chunks"][str(c)] = {**res, "verdict": "not available (the anchor's LAAL is outside the "
                                     "other policy's latency range in at least one language; no extrapolation)"}
            continue
        p = boot_p(bd)
        cd = ci(bd)
        out["chunks"][str(c)] = {**res, "delta": d, "ci95_unadjusted": cd, "p_boot": p,
                                 "resamples_outside_range": int((~np.isfinite(bd)).sum()),
                                 "equivalence": equivalence(cd, margin), "verdict": verdict(p, d)}
    return out


def table1(h3_fixed, h3_matched):
    """Table 1 rows (0.5 s, 1.0 s): WER and latency per policy and language, R, Delta, and the
    matched-latency Delta (review R1: latency is shown next to every WER used for H3)."""
    rows = []
    for c, r in h3_fixed["chunks"].items():
        if not isinstance(r, dict):
            continue
        m = h3_matched["chunks"].get(c, {})
        m = m if isinstance(m, dict) else {}
        row = {"chunk_s": float(c)}
        for lang in ("en", "id"):
            row[f"WER_LA_{lang}"] = 100 * r["wer"][f"la_{lang}"]
            row[f"WER_AA_{lang}"] = 100 * r["wer"][f"aa_{lang}"]
            for mode in ("cu", "ca"):
                row[f"LAAL{mode}_LA_{lang}"] = r["latency"][f"laal_{mode}_la_{lang}"]
                row[f"LAAL{mode}_AA_{lang}"] = r["latency"][f"laal_{mode}_aa_{lang}"]
            row[f"R_{lang}"] = 100 * r[f"R_{lang}"]
        row["p_precondition_holm"] = r.get("precondition_holm")
        row["p_keeps_holm"] = r.get("keeps_holm")
        row["Delta_pts"] = 100 * r["delta"]
        row["Delta_ci_lo"], row["Delta_ci_hi"] = [100 * v for v in r["ci95_unadjusted"]]
        row["p_H3_holm"] = r.get("h3_holm")
        row["absDelta_pts"] = 100 * r["abs_delta"]
        row["absDelta_ci_lo"], row["absDelta_ci_hi"] = [100 * v for v in r["abs_delta_ci95"]]
        row["Delta_matched_pts"] = 100 * m["delta"] if "delta" in m else np.nan
        lo, hi = m.get("ci95_unadjusted", [np.nan, np.nan])
        row["Delta_matched_ci_lo"], row["Delta_matched_ci_hi"] = 100 * lo, 100 * hi
        row["verdict_fixed"] = r.get("verdict")
        row["verdict_matched"] = m.get("verdict", "not available")
        row["equivalence_fixed"] = r["equivalence"]["verdict"]
        rows.append(row)
    return pd.DataFrame(rows)


def offline_check(df, manifest, B=10000, seed=20261022):
    """large-v2 (beam 5 + fallback) on the full test sets vs Radford et al. (2023), Table 11."""
    published = {"id": 7.1, "en": 4.2}
    out = {}
    rng = np.random.default_rng(seed)
    for lang in ("en", "id"):
        d = df[(df.policy == "offline_v2") & (df.lang == lang)]
        if d.empty:
            continue
        g = d.assign(E=d.S + d.D + d.I).groupby("idx")[["E", "N"]].sum()
        E, N = g.E.to_numpy(float), g.N.to_numpy(float)
        W = rng.multinomial(len(E), np.full(len(E), 1 / len(E)), size=B)
        wb = 100 * (W @ E) / (W @ N)
        lo, hi = ci(wb)
        out[lang] = {"wer": 100 * E.sum() / N.sum(), "ci95": [lo, hi], "published": published[lang],
                     "pass": bool(lo <= published[lang] <= hi), "recordings": int(len(d))}
    return out


def summary_table(df):
    """Pooled WER and mean latency per policy, language, chunk (CU text; CA latency)."""
    rows = []
    for (pol, lang, chunk), d in df[df["mode"] == "cu"].groupby(["policy", "lang", "chunk"], dropna=False):
        ca = df[(df.policy == pol) & (df.lang == lang) & (df["mode"] == "ca") & np.isclose(df.chunk, chunk)]
        N = d.N.sum()
        rows.append({
            "policy": pol, "lang": lang, "chunk": chunk, "recordings": len(d),
            "WER": 100 * (d.S + d.D + d.I).sum() / N, "sub": 100 * d.S.sum() / N,
            "del": 100 * d.D.sum() / N, "ins": 100 * d.I.sum() / N,
            "CER": 100 * d.CE.sum() / d.CN.sum(), "splits_per_100w": 100 * d.splits.sum() / N,
            "merges_per_100w": 100 * d.merges.sum() / N,
            "LAAL_cu": d.laal.mean(), "LAAL_ca": ca.laal.mean() if len(ca) else np.nan,
            "first_cu": d["first"].mean(), "first_ca": ca["first"].mean() if len(ca) else np.nan,
            "RTF": d.compute.sum() / d.dur.sum(),
            "erasure": d.erasure.mean(), "erasure_raw": d.erasure_raw.mean(),
            "tail_conf": d.tail_conf.sum() / d.tail_total.sum() if d.tail_total.sum() else np.nan,
            "spread": d["spread"].mean() if "spread" in d else np.nan,
            "share_lagging": ca.lag.mean() if len(ca) else np.nan,
        })
    for (pol, lang), d in df[df["mode"] == "offline"].groupby(["policy", "lang"]):
        N = d.N.sum()
        rows.append({"policy": pol, "lang": lang, "chunk": np.nan, "recordings": len(d),
                     "WER": 100 * (d.S + d.D + d.I).sum() / N, "sub": 100 * d.S.sum() / N,
                     "del": 100 * d.D.sum() / N, "ins": 100 * d.I.sum() / N,
                     "CER": 100 * d.CE.sum() / d.CN.sum(), "splits_per_100w": 100 * d.splits.sum() / N,
                     "merges_per_100w": 100 * d.merges.sum() / N,
                     "RTF": d.compute.sum() / d.dur.sum()})
    return pd.DataFrame(rows).sort_values(["lang", "policy", "chunk"])

In [ ]:
%%writefile /kaggle/working/wspb/qa.py
"""QA before the main runs (planned): the harness reproduces the released tools.

qa_localagreement  The LocalAgreement-2 port commits exactly what Whisper-Streaming's own
                   HypothesisBuffer (commit 6da90b4) commits when both receive the same decodes.
qa_alignatt        The AlignAtt engine wrapper outputs exactly what SimulStreaming's own command
                   line (commit 077ea37, --comp_unaware) outputs for the same file and settings.
"""
import json
import os
import subprocess
import sys

import numpy as np

from .commit import LocalAgreement
from .common import SR


def _hypothesis_buffer(ws_dir):
    if ws_dir not in sys.path:
        sys.path.insert(0, ws_dir)
    from whisper_online import HypothesisBuffer
    return HypothesisBuffer(logfile=open(os.devnull, "w"))


def qa_localagreement(engine, audio, chunk, ws_dir):
    """Same decodes into both implementations; returns (passed, n_steps, first difference)."""
    ours, ref = LocalAgreement(), _hypothesis_buffer(ws_dir)
    dur = len(audio) / SR
    end, k = 0.0, 0
    while True:
        end = min(end + chunk, dur)
        hyp = engine.decode(audio[: int(round(end * SR))])
        c_ours, _ = ours.step(hyp)
        ref.insert(hyp, 0.0)
        c_ref = ref.flush()
        if [tuple(w) for w in c_ours] != [tuple(w) for w in c_ref]:
            return False, k, {"step": k, "end": end, "ours": c_ours, "whisper_streaming": c_ref}
        k += 1
        if end >= dur - 1e-9:
            break
    # finish: Whisper-Streaming commits the remaining buffer (OnlineASRProcessor.finish -> complete())
    if [tuple(w) for w in ours.finish()] != [tuple(w) for w in ref.buffer]:
        return False, k, {"step": "finish"}
    return True, k, None


def qa_alignatt(engine_factory, wav_path, audio, lang, chunk, trim, ss_dir, model_path):
    """Run SimulStreaming's CLI on the file and compare its emitted text with the wrapper's
    committed pieces, including the final finish() call that the CLI makes after its loop."""
    cmd = [sys.executable, "simulstreaming_whisper.py", wav_path, "--comp_unaware",
           "--lan", lang, "--model_path", model_path, "--min-chunk-size", str(chunk),
           "--frame_threshold", "25", "-l", "WARNING"]
    if trim:
        cmd.append("--never_fire")
    p = subprocess.run(cmd, cwd=ss_dir, capture_output=True, text=True)
    if p.returncode != 0:
        return False, {"error": p.stderr[-2000:]}
    cli = []
    for line in p.stdout.splitlines():
        line = line.strip()
        if line.startswith("{"):
            cli.append(json.loads(line).get("text", ""))
    eng = engine_factory()
    try:
        eng.start()
        pieces, prev_end, dur = [], 0.0, len(audio) / SR
        # the CLI's own loop: beg = previous end, end += chunk, slices with int(t * 16000)
        end = chunk
        while True:
            text, _, _ = eng.step(audio[int(prev_end * SR): int(end * SR)])
            pieces.append(text)
            if end >= dur:
                break
            prev_end = end
            end = dur if end + chunk > dur else end + chunk
        text, _, _ = eng.finish()
        pieces.append(text)
    finally:
        eng.close()
    a, b = "".join(cli), "".join(pieces)
    return a == b, {"cli": a, "wrapper": b}


def run_all(cfg, manifest, n_per_lang=2, chunk=1.0):
    from . import engines
    from .data import load_audio

    ws_dir, ss_dir = cfg["whisper_streaming_dir"], cfg["simulstreaming_dir"]
    model = cfg.get("model", "large-v3")
    model_path = os.path.join(cfg["model_dir"], model + ".pt")
    report = {"localagreement": [], "alignatt": []}
    for lang in ("en", "id"):
        recs = [r for r in manifest["paired"] if r["lang"] == lang][:n_per_lang]
        eng = engines.RedecodeEngine(model, lang, cfg["model_dir"])
        try:
            for r in recs:
                ok, n, diff = qa_localagreement(eng, load_audio(r["path"]), chunk, ws_dir)
                report["localagreement"].append({"key": r["key"], "passed": ok, "steps": n, "diff": diff})
        finally:
            eng.close()
        for r in recs[:1]:
            for trim in (False, True):
                ok, detail = qa_alignatt(
                    lambda: engines.AlignAttEngine(model, lang, cfg["model_dir"], chunk,
                                                   frame_threshold=cfg["frame_threshold"], trim=trim),
                    r["path"], load_audio(r["path"]), lang, chunk, trim, ss_dir, model_path)
                report["alignatt"].append({"key": r["key"], "trim": trim, "passed": ok, **detail})
    report["passed"] = all(x["passed"] for v in ("localagreement", "alignatt") for x in report[v])
    return report

In [ ]:
%%writefile /kaggle/working/wspb/budget.py
"""GPU-time estimate for the full job list, from a check run on a few sentence indices."""
from pathlib import Path

import pandas as pd

from .common import read_jsonl


def _mode_factor(job, m):
    """A check job that lagged ran CA and then CU again; a single-mode job runs about half of that."""
    modes = job.get("modes", ("ca", "cu"))
    if len(modes) == 1 and (m["job"].get("modes") is None or len(m["job"]["modes"]) == 2):
        return 1 - 0.5 * (m["lag_share"] or 0)
    return 1.0


def estimate(check_results_dir, check_jobs, full_jobs, manifest, n_gpus=1, load_s=60.0):
    """Wall seconds per audio second of each check job, scaled to the audio of the full job.
    A full job with no check counterpart (same kind, model, beam) uses the nearest chunk size."""
    measured = {}
    for job in check_jobs:
        lines = read_jsonl(Path(check_results_dir) / f"{job['id']}.jsonl")
        if not lines:
            continue
        audio = sum(l["dur"] for l in lines)
        wall = sum(l.get("wall", 0.0) for l in lines)
        measured[job["id"]] = {"job": job, "s_per_audio_s": wall / audio, "lag_share":
                               sum(bool((l.get("ca") or {}).get("lag")) for l in lines) / len(lines)}

    def audio_seconds(job):
        recs = manifest["full"] if job.get("set") == "full" else manifest["paired"]
        ids = set(job["ids"]) if job.get("ids") is not None else None
        return sum(r["dur"] for r in recs if r["lang"] == job["lang"] and (ids is None or r["idx"] in ids))

    def match(job):
        if job["id"] in measured:
            return measured[job["id"]]
        cands = [m for m in measured.values()
                 if m["job"]["kind"] == job["kind"] and m["job"]["lang"] == job["lang"]
                 and m["job"].get("model") == job.get("model") and m["job"].get("beam", 1) == job.get("beam", 1)]
        if not cands:
            return None
        return min(cands, key=lambda m: abs(m["job"].get("chunk", 0) - job.get("chunk", 0)))

    rows = []
    for job in full_jobs:
        m = match(job)
        a = audio_seconds(job)
        rows.append({"job": job["id"], "audio_h": a / 3600,
                     "measured_from": m["job"]["id"] if m else None,
                     "s_per_audio_s": m["s_per_audio_s"] if m else None,
                     "ca_lag_share": m["lag_share"] if m else None,
                     "est_gpu_h": (m["s_per_audio_s"] * a * _mode_factor(job, m) + load_s) / 3600 if m else None})
    t = pd.DataFrame(rows)
    total = t.est_gpu_h.sum()
    return t, {"total_gpu_hours": float(total), "wall_hours_with_gpus": float(total / n_gpus),
               "jobs_without_estimate": t[t.est_gpu_h.isna()].job.tolist()}

In [ ]:
%%writefile /kaggle/working/wspb/plan.py
"""Job list in priority order (the runs the planned tests need come first).

Budget decision after the check run (22 Oct 2026): on a T4, large-v3 in fp32 runs slower than real
time at chunks <= 1.2 s, so the computation-aware (CA) simulation always lags and every job had to
be run twice. Main jobs therefore run computation-unaware (CU) only on all paired indices; CA
latency is measured on an evenly spaced subset of indices (`ca_ids`) at the H3 and Table 2 chunk
sizes. The anchor (real-time capable) keeps both modes. Beam-5 sensitivity runs at 1.0 s only.
"""


def ca_subset(paired_ids, n=40):
    """Evenly spaced subset of the paired sentence indices for the CA timing jobs."""
    ids = sorted(paired_ids)
    if len(ids) <= n:
        return ids
    step = len(ids) / n
    return [ids[int(i * step)] for i in range(n)]


def make_jobs(model="large-v3", v2_model="large-v2", chunks=(0.5, 1.0, 1.2, 1.5, 2.0),
              run_anchor=True, langs=("id", "en"), beam_chunks=(1.0,), ca_ids=None,
              ca_chunks=(1.0, 1.2)):
    def rd(lang, c):
        return {"id": f"redecode_{lang}_{c:g}", "kind": "redecode", "lang": lang, "chunk": c,
                "model": model, "set": "paired", "modes": ["cu"]}

    def aa(lang, c, trim):
        v = "aa_trim" if trim else "aa_keep"
        return {"id": f"{v}_{lang}_{c:g}", "kind": "aa", "lang": lang, "chunk": c, "model": model,
                "trim": trim, "policy_name": v, "set": "paired", "modes": ["cu"]}

    def off(lang, name, m, beam, s):
        return {"id": f"{name}_{lang}", "kind": "offline", "lang": lang, "model": m, "beam": beam,
                "policy_name": name, "set": s}

    def rd_beam(lang, c):
        # LocalAgreement (and hold-n) with beam 5 inside the harness: sensitivity check for the
        # greedy port (review R3). Policy names get the suffix "_beam5".
        return {"id": f"redecode_beam5_{lang}_{c:g}", "kind": "redecode", "lang": lang, "chunk": c,
                "model": model, "beam": 5, "policy_suffix": "_beam5", "set": "paired", "modes": ["cu"]}

    def ca_timing(job):
        # the same run, computation-aware only, on the CA subset; its rows are used for latency only
        return dict(job, id="catime_" + job["id"], modes=["ca"], ca_timing=True, ids=list(ca_ids))

    jobs = []
    tiers = [
        # tier 1: H3 (0.5, 1.0) and H4 (1.0 vs 2.0) for LocalAgreement and both AlignAtt variants
        [(c, k) for c in (1.0, 0.5, 2.0) for k in ("rd", "keep", "trim")],
        # tier 2: offline ceiling (M3 control) and setup check
        [("off", None)],
        # tier 2b: LocalAgreement beam-5 sensitivity (review R3)
        [(c, "beam") for c in (1.0, 0.5)],
        # tier 3: the shared 1.2 s chunk (RQ1, Table 2) and the 1.5 s curve point
        [(c, k) for c in (1.2, 1.5) for k in ("rd", "keep", "trim")],
    ]
    for tier in tiers:
        for c, k in tier:
            for lang in langs:
                if k is None:
                    jobs.append(off(lang, "offline_v3", model, False, "paired"))
                    jobs.append(off(lang, "offline_v2", v2_model, True, "full"))
                elif k == "beam":
                    if c in beam_chunks:
                        jobs.append(rd_beam(lang, c))
                elif c in chunks:
                    jobs.append(rd(lang, c) if k == "rd" else aa(lang, c, k == "trim"))
    # tier 4: computation-aware latency on the subset
    if ca_ids:
        jobs += [ca_timing(j) for j in list(jobs)
                 if j["kind"] in ("redecode", "aa") and "beam" not in j and j["chunk"] in ca_chunks]
    if run_anchor:
        for lang in langs:
            jobs.append({"id": f"ws_anchor_{lang}_1", "kind": "anchor", "lang": lang, "chunk": 1.0,
                         "model": model, "policy_name": "ws_anchor", "set": "paired",
                         "modes": ["ca", "cu"]})
    return jobs


# Seconds of T4 compute per second of audio, measured in the check run (22 Oct 2026; CA+CU where
# the job lagged). Only used to split the jobs into shards of similar length; relative values matter.
_SPEED = {
    ("redecode", 1): {0.5: 6.5, 1.0: 3.7, 1.2: 3.4, 1.5: 3.0, 2.0: 2.55},
    ("redecode", 5): {0.5: 9.5, 1.0: 5.2},
    ("aa_keep", 1): {0.5: 4.8, 1.0: 3.7, 1.2: 1.7, 1.5: 0.9, 2.0: 0.76},
    ("aa_trim", 1): {0.5: 3.55, 1.0: 2.45, 1.2: 2.2, 1.5: 1.3, 2.0: 0.8},
    ("anchor", 1): {1.0: 1.2},
}
_AUDIO_H = {"id": 2.31, "en": 1.64}          # paired audio per language
_N_PAIRED = 325


def job_cost(job):
    """Rough relative cost of a job (T4 hours), for balancing shards."""
    if job["kind"] == "offline":
        return _AUDIO_H[job["lang"]] * (0.36 if job.get("beam") else 0.17)
    key = (job.get("policy_name") if job["kind"] == "aa" else job["kind"], job.get("beam", 1))
    speeds = _SPEED.get(key, {})
    s = speeds.get(job.get("chunk"), max(speeds.values()) if speeds else 3.0)
    if job.get("modes") == ["cu"] or job.get("modes") == ["ca"]:
        s *= 0.5                                # one mode instead of CA + CU
    frac = len(job["ids"]) / _N_PAIRED if job.get("ids") is not None else 1.0
    return _AUDIO_H[job["lang"]] * s * frac


def shard_jobs(jobs, n_shards, index):
    """Split the jobs over n_shards accounts/machines: each job goes, in priority order, to the
    shard with the least work so far, so every shard starts with high-priority jobs and the
    shards take about the same time. Deterministic: every account computes the same split."""
    if n_shards <= 1:
        return list(jobs)
    load = [0.0] * n_shards
    mine = []
    for job in jobs:
        k = min(range(n_shards), key=lambda i: (load[i], i))
        load[k] += job_cost(job)
        if k == index:
            mine.append(job)
    return mine

In [ ]:
%%writefile /kaggle/working/wspb/report.py
"""Run every analysis on whatever has finished, and write tables, Figure 1 and a summary."""
import json
from pathlib import Path

import numpy as np

from . import stats
from .common import save_json
from .data import index_features
from .metrics import collect

LABEL = {"hold0": "Hold-0", "hold2": "Hold-2", "la2": "LocalAgreement-2",
         "la2_beam5": "LocalAgreement-2, beam 5 (sensitivity)",
         "aa_keep": "AlignAtt-keep (SimulStreaming default)", "aa_trim": "AlignAtt-trim",
         "ws_anchor": "Whisper-Streaming as released", "offline_v3": "Offline large-v3 (greedy)"}


def _safe(fn, *a, **k):
    """One analysis failing (e.g. too few indices in a check run) must not stop the others."""
    try:
        return fn(*a, **k)
    except Exception as e:  # noqa: BLE001 - reported in stats.json
        return {"error": f"{type(e).__name__}: {e}"}


def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o


def _md(t):
    try:
        return t.to_markdown(index=False)
    except ImportError:
        return t.to_string(index=False)


def figure1(table, out_dir, primary):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    pols = ["hold0", "hold2", "la2", "aa_keep", "aa_trim"]
    markers = {"hold0": "s", "hold2": "D", "la2": "o", "aa_keep": "^", "aa_trim": "v"}
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0), sharey=True)
    for ax, lang, title in zip(axes, ("en", "id"), ("English", "Indonesian")):
        t = table[table.lang == lang]
        for p in pols:
            d = t[t.policy == p].sort_values("chunk")
            if d.empty:
                continue
            ax.plot(d.LAAL_cu, d.WER, marker=markers[p], ms=4, lw=1.4 if p in ("la2", primary) else 1.0,
                    label=LABEL[p])
            for _, r in d.iterrows():
                ax.annotate(f"{r.chunk:g}", (r.LAAL_cu, r.WER), fontsize=6, xytext=(2, 2),
                            textcoords="offset points")
        off = t[t.policy == "offline_v3"]
        if len(off):
            ax.axhline(off.WER.iloc[0], color="0.5", ls="--", lw=0.8, label=LABEL["offline_v3"])
        anc = t[t.policy == "ws_anchor"]
        if len(anc):
            ax.plot(anc.LAAL_cu, anc.WER, "o", mfc="none", mec="k", ms=6, label=LABEL["ws_anchor"])
        ax.set_title(title, fontsize=9)
        ax.set_xlabel("LAAL, computation-unaware (s)", fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.3, lw=0.5)
    axes[0].set_ylabel("WER (%)", fontsize=8)
    h, l = axes[1].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=3, fontsize=6.5, frameon=False, bbox_to_anchor=(0.5, -0.08))
    fig.tight_layout(rect=(0, 0.1, 1, 1))
    for ext in ("png", "pdf"):
        fig.savefig(Path(out_dir) / f"figure1_wer_laal.{ext}", dpi=300, bbox_inches="tight")
    plt.close(fig)


def run(cfg, manifest, speech, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    df = collect(cfg["results_dir"], cfg["jobs"], manifest)
    if df.empty:
        print("No results yet.")
        return None
    df.drop(columns=["text"]).to_csv(out_dir / "per_recording_metrics.csv", index=False)
    table = stats.summary_table(df)
    table.to_csv(out_dir / "summary_by_run.csv", index=False)

    primary = cfg["primary_aa"]
    ids = manifest["paired_ids"]
    if cfg.get("quick_ids") is not None:
        ids = [i for i in ids if i in set(cfg["quick_ids"])]
    D = stats.Data(df, ids, B=cfg["bootstrap_B"], seed=cfg["seed"])
    spw = index_features(manifest, speech)
    streaming = ["hold0", "hold2", "la2", primary]
    margin = cfg.get("equivalence_margin", 0.42)
    h3_primary = _safe(stats.h3, D, primary, margin=margin)
    h3_matched_cu = _safe(stats.h3_matched_latency, D, primary, lat_mode="cu", anchor="la", margin=margin)
    h3_matched_ca = _safe(stats.h3_matched_latency, D, primary, lat_mode="ca", anchor="la", margin=margin)
    h3_matched_rev = _safe(stats.h3_matched_latency, D, primary, lat_mode="cu", anchor="aa", margin=margin)
    res = {
        "primary_alignatt": primary,
        "n_indices": len(ids),
        "equivalence_margin": margin,
        "H3_primary": h3_primary,
        "H3_matched_latency_primary_cu": h3_matched_cu,
        "H3_matched_latency_secondary_ca": h3_matched_ca,
        "H3_matched_latency_secondary_reverse_anchor": h3_matched_rev,
        "H3_CER_secondary": _safe(stats.h3, D, primary, metric="cer", margin=margin),
        "H3_equal_index_weight_sensitivity": _safe(stats.h3, D, primary, equal_weight=True, margin=margin),
        "H3_LA_beam5_sensitivity": _safe(stats.h3, D, primary, la="la2_beam5", margin=margin),
        "H3_other_alignatt": _safe(stats.h3, D, "aa_trim" if primary == "aa_keep" else "aa_keep"),
        "H3_without_number_indices": _safe(stats.h3, D, primary, exclude=manifest["number_ids"]),
        "H4_alignatt_primary": _safe(stats.h4, D, primary),
        "H4_localagreement_secondary": _safe(stats.h4, D, "la2"),
        "H4_without_number_indices": _safe(stats.h4, D, primary, exclude=manifest["number_ids"]),
        "H1_ranking": _safe(stats.h1, D, streaming, [0.5, 1.0, 1.2]),
        "H2_language_gap": _safe(stats.h2, D, streaming, [0.5, 1.0, 1.2, 2.0]),
        "H5_alignatt_latency": _safe(stats.h5, D, primary, [0.5, 1.0, 1.2, 1.5, 2.0], mode="cu"),
        "H5_alignatt_latency_ca_subset": _safe(stats.h5, D, primary, [1.0, 1.2], mode="ca"),
        "M3_word_duration_slope": _safe(stats.m3, D, spw, la="la2", aa=primary),
        "M4_held_back_word_excess": _safe(stats.m4, D, "aa_trim"),
        "M5_trim_gain": _safe(stats.m5_trim_gain, D),
        "offline_setup_check": _safe(stats.offline_check, df, manifest, B=cfg["bootstrap_B"], seed=cfg["seed"]),
    }
    res = _clean(res)
    save_json(res, out_dir / "stats.json")
    _safe(figure1, table, out_dir, primary)

    t1 = _safe(stats.table1, h3_primary, h3_matched_cu)
    if not hasattr(t1, "to_csv"):
        import pandas as pd
        t1 = pd.DataFrame([t1])
    t1.to_csv(out_dir / "table1_h3.csv", index=False)
    t2 = table[np.isclose(table.chunk.fillna(-1), 1.2) | (table.policy == "ws_anchor")]
    t2.to_csv(out_dir / "table2_metrics_1.2s.csv", index=False)
    md_table = _md(table.round(3))
    lines = ["# Results summary", "",
             f"Primary AlignAtt variant: **{LABEL.get(primary, primary)}**. Text metrics: computation-unaware runs; "
             "latency: computation-unaware LAAL (all indices; primary) and computation-aware LAAL on the virtual "
             "clock (CA timing subset of indices; secondary).", "",
             "## Table 1 (H3)", "", _md(t1.round(3).T.reset_index()), "",
             "## H3 (main result)", "```", json.dumps(res["H3_primary"], indent=1)[:6000], "```", "",
             "## H3 at matched latency (computation-unaware LAAL, primary)", "```",
             json.dumps(res["H3_matched_latency_primary_cu"], indent=1)[:4000], "```", "",
             "## Offline setup check", "```", json.dumps(res["offline_setup_check"], indent=1), "```", "",
             "## WER and latency by run", "", md_table, ""]
    (out_dir / "summary.md").write_text("\n".join(lines), encoding="utf-8")
    return res, table

## Setup, unit tests, restore

In [ ]:
import glob, json, shutil, importlib
for p in (OUT, SS_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)
import wspb
from wspb import common, text, align, data, commit, plan, report, qa, budget, stats
for m in (common, text, align, data, commit, plan, report, qa, budget, stats):
    importlib.reload(m)

assert commit.self_test() and text.self_test()
print("unit tests passed")

ENV = common.env_info()
common.save_json(ENV, f"{STATE}/environment_{MODE}.json")
print(json.dumps(ENV, indent=1))

import torch
N_GPUS = max(1, torch.cuda.device_count())
print("GPUs:", N_GPUS)

# Restore finished work from previous versions attached as input (Add Input -> Notebook Output)
restored = 0
if MODE == "run":
    for src in glob.glob("/kaggle/input/**/results/*.jsonl", recursive=True):
        dst = os.path.join(RESULTS, os.path.basename(src))
        n_src = sum(1 for _ in open(src, encoding="utf-8"))
        n_dst = sum(1 for _ in open(dst, encoding="utf-8")) if os.path.exists(dst) else -1
        if n_src > n_dst:
            shutil.copy(src, dst); restored += 1
    for src in glob.glob("/kaggle/input/**/results/*.done", recursive=True):
        shutil.copy(src, os.path.join(RESULTS, os.path.basename(src)))
for src in glob.glob("/kaggle/input/**/state/*.json", recursive=True):
    dst = os.path.join(STATE, os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copy(src, dst)
shutil.rmtree(f"{RESULTS}/claims", ignore_errors=True)   # locks of a previous session
print("restored result files:", restored)

## Data: FLEURS test sets, paired manifest, VAD

In [ ]:
t = time.time()
data.fetch_fleurs(DATA_DIR)
manifest = data.build_manifest(DATA_DIR)
print("paired indices:", len(manifest["paired_ids"]), "| excluded (>30 s):", manifest["excluded_too_long"],
      "| indices with numbers:", len(manifest["number_ids"]))

speech_path = f"{STATE}/speech_durations.json"
speech = data.speech_durations(manifest, speech_path, VAD_PARAMS)
stats_data = data.data_statistics(manifest, speech)
common.save_json(stats_data, f"{STATE}/data_statistics.json")
print(json.dumps(stats_data, indent=1))
print(f"data ready in {(time.time() - t) / 60:.1f} min")

## Models

In [ ]:
import whisper as openai_whisper
t = time.time()
for name in (MODEL, "large-v2"):
    path = os.path.join(MODEL_DIR, name + ".pt")
    if not os.path.exists(path):
        openai_whisper._download(openai_whisper._MODELS[name], MODEL_DIR, in_memory=False)
    print(name, os.path.getsize(path) // 2**20, "MB")
print(f"models ready in {(time.time() - t) / 60:.1f} min")

## QA: harness vs released tools

In [ ]:
# QA before the main runs (planned): the harness reproduces the released tools.
BASE_CFG = dict(simulstreaming_dir=SS_DIR, whisper_streaming_dir=WS_DIR, data_dir=DATA_DIR,
                model_dir=MODEL_DIR, model=MODEL, frame_threshold=FRAME_THRESHOLD, hold_n=HOLD_N,
                primary_aa=PRIMARY_AA, bootstrap_B=BOOTSTRAP_B, seed=SEED,
                equivalence_margin=EQUIVALENCE_MARGIN)
qa_path = f"{STATE}/qa.json"
if os.path.exists(qa_path):
    QA_RES = common.load_json(qa_path)
    print("QA already done in an earlier session")
else:
    t = time.time()
    QA_RES = qa.run_all(BASE_CFG, manifest)
    common.save_json(QA_RES, qa_path)
    print(f"QA took {(time.time() - t) / 60:.1f} min")
for kind in ("localagreement", "alignatt"):
    for r in QA_RES[kind]:
        extra = "" if r["passed"] else " <-- MISMATCH: " + json.dumps(
            {k: v for k, v in r.items() if k not in ("passed",)}, ensure_ascii=False)[:1500]
        print(f"QA {kind:15s} {r['key']:40s} trim={r.get('trim', '-')!s:5s} passed={r['passed']}{extra}")
print("QA PASSED" if QA_RES["passed"] else "QA FAILED (see mismatches above)")

## Runs

In [ ]:
CA_IDS = plan.ca_subset(manifest["paired_ids"], n=40)   # CA timing subset (budget decision)
ALL_JOBS = plan.make_jobs(model=MODEL, ca_ids=CA_IDS)
JOBS = plan.shard_jobs(ALL_JOBS, N_SHARDS, SHARD_INDEX) if MODE == "run" else ALL_JOBS
print(f"shard {SHARD_INDEX + 1} of {N_SHARDS}: {len(JOBS)} of {len(ALL_JOBS)} jobs, "
      f"~{sum(plan.job_cost(j) for j in JOBS):.0f} T4 GPU-hours (rough)")
quick = manifest["paired_ids"][:CHECK_N_INDICES] if MODE == "check" else None
deadline = T0 + SESSION_HOURS * 3600 - REPORT_RESERVE_MIN * 60
cfg = dict(BASE_CFG, results_dir=RESULTS, jobs=JOBS, quick_ids=quick, deadline_epoch=deadline)
cfg_path = f"{RESULTS}/config.json"
common.save_json(cfg, cfg_path)
cfg_report = dict(cfg, jobs=ALL_JOBS)      # the report always covers every job that has results
os.makedirs(f"{RESULTS}/logs", exist_ok=True)
print(f"{len(JOBS)} jobs; {N_GPUS} worker(s); deadline in {(deadline - time.time()) / 3600:.2f} h")

env = dict(os.environ, PYTHONPATH=os.pathsep.join([OUT, SS_DIR]), PYTHONUNBUFFERED="1",
           LD_LIBRARY_PATH=os.pathsep.join(CUDNN + [os.environ.get("LD_LIBRARY_PATH", "")]))
procs = []
for g in range(N_GPUS):
    out_f = open(f"{RESULTS}/logs/worker{g}.stdout", "a")
    procs.append(subprocess.Popen([sys.executable, "-m", "wspb.worker", "--config", cfg_path,
                                   "--worker", str(g)], env=dict(env, CUDA_VISIBLE_DEVICES=str(g)),
                                  stdout=out_f, stderr=subprocess.STDOUT))
seen = {}
def show_new_log_lines():
    for g in range(N_GPUS):
        f = f"{RESULTS}/logs/worker{g}.log"
        if not os.path.exists(f):
            continue
        lines = open(f, encoding="utf-8").read().splitlines()
        for line in lines[seen.get(f, 0):]:
            print(f"[gpu{g}] {line}")
        seen[f] = len(lines)
while any(p.poll() is None for p in procs):
    time.sleep(60)
    show_new_log_lines()
show_new_log_lines()
print("worker exit codes:", [p.returncode for p in procs])
for g in range(N_GPUS):
    so = open(f"{RESULTS}/logs/worker{g}.stdout", encoding="utf-8", errors="replace").read()
    if "Traceback" in so:
        print(f"--- worker{g}.stdout (tail) ---"); print(so[-4000:])

In [ ]:
# Job status of ALL jobs: done / partial / not started, and which shard runs each job
from wspb.worker import records_for
mine_ids = {j["id"] for j in JOBS}
rows = []
for job in ALL_JOBS:
    n_have = len(common.read_jsonl(f"{RESULTS}/{job['id']}.jsonl"))
    n_need = len(records_for(job, manifest, set(quick) if quick else None))
    rows.append((job["id"], "this account" if job["id"] in mine_ids else "other account", n_have, n_need,
                 "done" if os.path.exists(f"{RESULTS}/{job['id']}.done") else ("partial" if n_have else "not started")))
import pandas as pd
st = pd.DataFrame(rows, columns=["job", "run_by", "recordings_saved", "recordings_needed", "status"])
print(st.to_string(index=False))
print("this account:", st[st.run_by == "this account"].status.value_counts().to_dict())
print("all jobs:    ", st.status.value_counts().to_dict())
if (st.status != "done").any():
    print("NOTE: the report below covers only the jobs with results so far.")

## Report

In [ ]:
import warnings
warnings.simplefilter("ignore", RuntimeWarning)
t = time.time()
rep = report.run(cfg_report, manifest, speech, REPORT)
if rep is not None:
    res, table = rep
    print(open(f"{REPORT}/summary.md", encoding="utf-8").read()[:12000])
print(f"report in {(time.time() - t) / 60:.1f} min; files:", sorted(os.listdir(REPORT)))

In [ ]:
# GPU-time estimate for the full plan (check mode only)
if MODE == "check":
    full_jobs = plan.make_jobs(model=MODEL, ca_ids=CA_IDS)
    bt, tot = budget.estimate(RESULTS, JOBS, full_jobs, manifest, n_gpus=N_GPUS)
    bt.to_csv(f"{REPORT}/gpu_budget.csv", index=False)
    print(bt.round(3).to_string(index=False))
    print(json.dumps(tot, indent=1))
    print(f"Kaggle sessions of {SESSION_HOURS} h needed (approx.): "
          f"{tot['wall_hours_with_gpus'] / SESSION_HOURS:.1f}")

In [ ]:
shutil.make_archive(f"{OUT}/outputs_{MODE}", "zip", OUT, os.path.basename(REPORT))
print(f"total session time {(time.time() - T0) / 3600:.2f} h")

## Download everything (results, logs, report, state, code)

In [ ]:
# Zip all outputs of this run and show a download link (also in the Output tab after a commit)
from IPython.display import FileLink, display
stamp = time.strftime("%Y%m%d_%H%M")
zip_base = f"{OUT}/wspb_{MODE}_shard{SHARD_INDEX}_all_{stamp}"
tmp = f"/tmp/wspb_export_{stamp}"
shutil.rmtree(tmp, ignore_errors=True)
os.makedirs(tmp)
for name in sorted(os.listdir(OUT)):
    src = os.path.join(OUT, name)
    if name.endswith(".zip") or name.startswith("."):
        continue
    (shutil.copytree if os.path.isdir(src) else shutil.copy2)(src, os.path.join(tmp, name))
zip_path = shutil.make_archive(zip_base, "zip", tmp)
shutil.rmtree(tmp, ignore_errors=True)
print("Created:", zip_path, f"({os.path.getsize(zip_path) / 2**20:.1f} MB)")
os.chdir(OUT)
display(FileLink(os.path.basename(zip_path)))